# Raw player-season and game data

Downloads player-season tables from Baseball Savant, FanGraphs and the MLB Stats API into `data/raw/` and validates them. The Savant custom-leaderboard and percentile pulls build on Jeffrey's earlier scraping cells.

Reruns re-validate and reuse finished files. With every file present and `OVERWRITE = False`, a rerun makes no network requests. If a saved file fails validation, the run stops: inspect the file, then delete it or set `OVERWRITE = True`.

**Leakage:** every row is a full-season total. Don't use a season's row as pre-game information for games in that same season.

The second part, **Game-level data**, collects one record per regular-season game for 2022-2026 into `data/games/`: schedules and official results, starting lineups and pitchers, bullpen membership, and 2022-2025 play records. Raw game responses are cached outside the repository.

## Imports

In [1]:
import gzip
import json
import os
import random
import re
import time
from collections import Counter
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from pandas.api.types import union_categoricals

# Request sources directly, not through pybaseball, so responses can be checked before saving.

## Configuration

Each source defines its minimum differently: plate appearances on some boards, innings on others. The comments give each meaning, checked with small live requests.

All minimums are set to 0. Source coverage still varies. Apply qualification filters during modeling.

In [2]:
SEASONS = [2022, 2023, 2024, 2025, 2026]
# Find the repository root so data/raw/ is the same whether Jupyter starts there or in notebooks/.
REPO_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                  if (folder / "requirements.txt").is_file()
                  and (folder / "notebooks" / "savant_scrape_updated.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"Could not find the repository root from {Path.cwd().name}/: no folder above it contains "
                            "requirements.txt and notebooks/savant_scrape_updated.ipynb. Start Jupyter inside the repository.")
RAW_DIR = REPO_ROOT / "data" / "raw"   # tracked in Git; *.partial files are ignored
OVERWRITE = False                # False: re-validate and reuse finished files

# Savant custom, batters: min = minimum PA. 0 and 1 return the same players.
SAVANT_CUSTOM_MIN_BATTER = 0
# Savant custom, pitchers: min = minimum innings, not PA. min=1 drops pitchers with 0.1-0.2 IP
# (7 in a 2026 check).
SAVANT_CUSTOM_MIN_PITCHER = 0
# Savant expected stats: min = minimum PA, for batters and pitchers.
SAVANT_XSTATS_MIN_PA = 0
# FanGraphs: qual = minimum PA (batting) or IP (pitching). At 0 the batting board also lists pitchers with 0 PA.
FANGRAPHS_QUAL = 0
# Savant percentile rankings take no minimum; Savant ranks only qualified players.

REQUEST_PAUSE_SECONDS = 2        # pause after each request; requests run one at a time
HEADERS = {"User-Agent": "mlb-game-simulator-research-notebook (python-requests)"}

RAW_DIR.mkdir(parents=True, exist_ok=True)
# Game-level collection (see "Game-level data")
GAME_SEASONS = [2022, 2023, 2024, 2025, 2026]
PLAY_SEASONS = [2022, 2023, 2024, 2025]   # development seasons: detailed play records are collected only here
GAME_DIR = REPO_ROOT / "data" / "games"   # game-level CSVs, kept apart from the season tables in data/raw/
# Raw MLB JSON and Savant CSV responses are caches. They stay outside the repository so Git never sees them;
# set MLB_GAME_CACHE_DIR to move them.
GAME_CACHE_DIR = Path(os.environ.get("MLB_GAME_CACHE_DIR", Path.home() / ".cache" / "mlb_game_simulator"))
# None collects every scheduled game. A list of gamePks collects only those games, plus each team's previous
# game for bullpens, and writes to GAME_CACHE_DIR / "sample_output" instead of data/games/.
GAME_SAMPLE = None
OVERWRITE_GAMES = False          # False: an existing game CSV must match the rebuilt table, or the run stops
MAX_CSV_MB = 50                  # GitHub warns above 50 MB and rejects files above 100 MB
ROSTER_CHECKS_PER_SEASON = 30    # team-games per season used to test the dated-roster endpoint

print("raw data folder:", RAW_DIR.relative_to(Path.cwd(), walk_up=True))

raw data folder: ..\data\raw


## Request and file helpers

- Every request has a timeout. Connection errors, timeouts, 429 and 5xx get up to three attempts and honor `Retry-After`. Other error statuses stop the run, so an access block is reported, not worked around.
- Savant can answer a bad request with an HTML page and status 200, so HTML and header-only CSVs are rejected.
- Each season table must have rows, valid player IDs, one row per player and only the requested season.
- Files are written to `.partial`, read back, then renamed. An interrupted run leaves only a `.partial` file.
- Saved files are re-validated before reuse. A file that fails stops the run instead of being replaced.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)
RETRY_STATUSES = {429, 500, 502, 503, 504}
requests_made = []      # stays empty on a rerun with every file saved
RAW_TABLES = []         # for the validation section


def fetch(url, params, source, max_attempts=3):
    """GET with a timeout and status check, retrying transient failures."""
    for attempt in range(1, max_attempts + 1):
        requests_made.append(source)
        try:
            response = session.get(url, params=params, timeout=(15, 180))
        except (requests.ConnectionError, requests.Timeout) as error:
            problem, wait = f"{type(error).__name__}: {error}", 10 * attempt
        else:
            if response.status_code < 400:
                time.sleep(REQUEST_PAUSE_SECONDS)
                return response
            problem = f"HTTP {response.status_code}"
            if response.status_code not in RETRY_STATUSES:
                raise RuntimeError(f"{source}: {problem} from {response.url[:200]}")
            retry_after = response.headers.get("Retry-After", "")
            wait = min(int(retry_after), 300) if retry_after.isdigit() else 10 * attempt
        if attempt == max_attempts:
            raise RuntimeError(f"{source}: failed after {max_attempts} attempts ({problem})")
        print(f"  {source}: {problem}; retrying in {wait} s")
        time.sleep(wait)


def read_csv_response(response, source):
    """Parse a Savant CSV, rejecting HTML pages and header-only files."""
    body = response.content
    content_type = response.headers.get("Content-Type", "")
    if "html" in content_type or body.lstrip()[:1] == b"<":
        raise ValueError(f"{source}: expected CSV but received HTML ({content_type}); starts {body[:80]!r}")
    table = pd.read_csv(StringIO(body.decode("utf-8-sig")))   # Savant CSVs begin with a byte-order mark
    if table.empty:
        raise ValueError(f"{source}: the CSV contained no player rows")
    return table


def check_table(table, source, season, id_column, season_column, required_columns=()):
    """Raise ValueError listing every structural problem in a season table."""
    problems = []
    if table.empty:
        problems.append("no rows")
    missing = [c for c in [id_column, season_column, *required_columns] if c not in table.columns]
    if missing:
        problems.append(f"missing columns {missing}")
    else:
        ids = pd.to_numeric(table[id_column], errors="coerce")
        if ids.isna().any():
            problems.append(f"{ids.isna().sum()} missing or non-numeric {id_column}")
        elif (ids % 1 != 0).any():
            problems.append(f"non-integer {id_column}")
        if table[id_column].duplicated().any():
            problems.append(f"{table[id_column].duplicated().sum()} duplicate {id_column} (expected one row per player-season)")
        seasons = sorted(pd.unique(table[season_column]).tolist())
        if seasons != [season]:
            problems.append(f"{season_column} values {seasons[:5]}, expected [{season}]")
    if problems:
        raise ValueError(f"{source} {season}: " + "; ".join(problems))


def save_csv(table, path):
    """Write to <name>.partial, check it reads back identically, then rename it."""
    partial = path.with_name(path.name + ".partial")
    table.to_csv(partial, index=False)
    assert_same_values(pd.read_csv(partial), table)
    partial.replace(path)


def assert_same_values(left, right):
    """Assert two tables match cell by cell. None and NaN count as equal; a CSV can't tell them apart."""
    as_objects = lambda t: t.reset_index(drop=True).astype(object).where(t.reset_index(drop=True).notna(), None)
    pd.testing.assert_frame_equal(as_objects(left), as_objects(right), check_dtype=False, check_exact=False)


def load_or_download(path, download, validate):
    """Re-validate and reuse a finished file, or download, validate and save it."""
    if path.exists() and not OVERWRITE:
        table = pd.read_csv(path)
        try:
            validate(table)
        except ValueError as error:
            raise ValueError(f"{path} exists but failed validation, so it was neither reused nor replaced: "
                             f"{error}. Inspect it, then delete it or set OVERWRITE = True.") from error
        print(f"  reused {path.name}: {table.shape[0]} rows x {table.shape[1]} columns")
        return table
    table = download()
    validate(table)
    save_csv(table, path)
    print(f"  saved  {path.name}: {table.shape[0]} rows x {table.shape[1]} columns")
    return pd.read_csv(path)    # continue with exactly what is on disk


def register(source, role, season, path, table, id_column):
    RAW_TABLES.append({"source": source, "role": role, "season": season, "file": path.name,
                       "rows": table.shape[0], "columns": table.shape[1], "id_column": id_column, "table": table})

# Data acquisition

Each season table has one row per player per season. Traded players appear once, with combined totals. The sources define and include players differently, so each table is saved separately and nothing is merged here.

| Source | Files (per season) | Player ID column | Season column |
|---|---|---|---|
| Baseball Savant custom leaderboard | `savant_custom_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| Baseball Savant expected statistics | `savant_expected_stats_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| Baseball Savant percentile rankings (supplementary) | `savant_percentile_ranks_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| FanGraphs major-league leaderboards | `fangraphs_{batting,pitching}_{season}.csv` | `playerid` (FanGraphs); `xMLBAMID` (MLBAM) | `Season` |
| MLB Stats API people lookup | `mlb_player_handedness.csv` (one file) | `player_id` (MLBAM) | not seasonal |

Join on MLBAM player ID and season: `player_id` and `year` in Savant, `xMLBAMID` and `Season` in FanGraphs. Each pair should be unique within a role and season, and validation checks this. Don't join on names: Savant writes "Last, First", FanGraphs writes "First Last", and names aren't unique.

## Savant custom leaderboard: batters

`https://baseballsavant.mlb.com/leaderboard/custom` with `csv=true` returns actual statistics. With `min=0` it lists every batter with at least one plate appearance. The CSV always adds `last_name, first_name`, `player_id` and `year`.

The field list is every batter field on the live page, grouped for readability. The 2022 and 2026 schemas matched when checked. `xera` and `r_stolen_base_pct` are left out because they were empty on the batter board in every season checked.

Units: `*_percent` and `*_rate` fields are percentages (24.4 means 24.4 %), speeds are mph, angles are degrees, and `avg_swing_length` is feet.

`pa` is Savant's Statcast-tracked plate-appearance count and `b_total_pa` is its standard count. They can differ by one or two, so validation compares FanGraphs' `PA` with both.

Bat tracking starts in 2023, so those fields are empty in 2022. Validation reports empty fields by season rather than filling them.

In [4]:
BATTER_FIELD_GROUPS = {
    "profile": ["player_age"],
    # Counts are kept with the rates so later work can re-aggregate.
    "results": ["pa", "ab", "hit", "single", "double", "triple", "home_run", "strikeout", "walk",
                "batting_avg", "slg_percent", "on_base_percent", "on_base_plus_slg", "isolated_power",
                "babip", "woba", "k_percent", "bb_percent"],
    # Savant's standard batting counts
    "batting_counts": ["b_ab_scoring", "b_ball", "b_called_strike", "b_catcher_interf", "b_foul",
                       "b_foul_tip", "b_game", "b_gnd_into_dp", "b_gnd_into_tp", "b_gnd_rule_double",
                       "b_hit_by_pitch", "b_hit_fly", "b_hit_ground", "b_hit_into_play", "b_hit_line_drive",
                       "b_hit_popup", "b_intent_ball", "b_intent_walk", "b_interference", "b_lob",
                       "b_out_fly", "b_out_ground", "b_out_line_drive", "b_out_popup", "b_pinch_hit",
                       "b_pinch_run", "b_pitchout", "b_played_dh", "b_rbi", "b_reached_on_error",
                       "b_reached_on_int", "b_sac_bunt", "b_sac_fly", "b_swinging_strike", "b_total_ball",
                       "b_total_bases", "b_total_pa", "b_total_pitches", "b_total_sacrifices",
                       "b_total_strike", "b_total_swinging_strike", "b_walkoff"],
    # x* = expected, from exit velocity and launch angle; *diff = actual minus expected
    "expected": ["xba", "xslg", "xwoba", "xobp", "xiso", "xwobacon", "xbacon", "wobacon", "bacon",
                 "xbadiff", "xslgdiff", "wobadiff"],
    "contact_quality": ["batted_ball", "exit_velocity_avg", "launch_angle_avg", "sweet_spot_percent",
                        "barrel", "barrel_batted_rate", "hard_hit_percent", "avg_best_speed",
                        "avg_hyper_speed", "solidcontact_percent", "flareburner_percent",
                        "poorlyunder_percent", "poorlytopped_percent", "poorlyweak_percent"],
    "batted_ball_profile": ["groundballs", "groundballs_percent", "flyballs", "flyballs_percent",
                            "linedrives", "linedrives_percent", "popups", "popups_percent",
                            "pull_percent", "straightaway_percent", "opposite_percent"],
    # z_ = in the strike zone, oz_ = outside it
    "plate_discipline": ["pitch_count", "swing_percent", "whiff_percent", "z_swing_percent",
                         "z_swing_miss_percent", "oz_swing_percent", "oz_swing_miss_percent",
                         "oz_contact_percent", "iz_contact_percent", "in_zone", "in_zone_percent",
                         "in_zone_swing", "in_zone_swing_miss", "out_zone", "out_zone_percent",
                         "out_zone_swing", "out_zone_swing_miss", "f_strike_percent", "edge",
                         "edge_percent", "meatball_percent", "meatball_swing_percent", "swords"],
    # from 2023
    "bat_tracking": ["avg_swing_speed", "avg_swing_length", "squared_up_contact", "squared_up_swing",
                     "blasts_contact", "blasts_swing", "fast_swing_rate", "attack_angle",
                     "attack_direction", "ideal_angle_rate", "vertical_swing_path"],
    # run values are in runs
    "pitch_group_run_value": ["pitch_count_fastball", "pitch_count_breaking", "pitch_count_offspeed",
                              "pitch_run_value_fastball", "pitch_run_value_breaking",
                              "pitch_run_value_offspeed", "swing_take_run_value"],
    "baserunning": ["sprint_speed", "hp_to_1b", "n_bolts", "runner_run_value", "r_run",
                    "r_caught_stealing_2b", "r_caught_stealing_3b", "r_caught_stealing_home",
                    "r_defensive_indiff", "r_interference", "r_pickoff_1b", "r_pickoff_2b",
                    "r_pickoff_3b", "r_stolen_base_2b", "r_stolen_base_3b", "r_stolen_base_home",
                    "r_total_caught_stealing", "r_total_pickoff", "r_total_stolen_base"],
    # Position-specific defense (catcher pop times, outfielder jumps), mostly empty for other players.
    # Kept in the raw data only.
    "defense_ancillary": ["n_outs_above_average", "fielding_run_value", "n_1star_percent",
                          "n_2star_percent", "n_3star_percent", "n_4star_percent", "n_5star_percent",
                          "n_fieldout_1stars", "n_fieldout_2stars", "n_fieldout_3stars",
                          "n_fieldout_4stars", "n_fieldout_5stars", "n_opp_1stars", "n_opp_2stars",
                          "n_opp_3stars", "n_opp_4stars", "n_opp_5stars", "f_bootup_distance",
                          "rel_league_bootup_distance", "rel_league_burst_distance",
                          "rel_league_reaction_distance", "rel_league_routing_distance",
                          "pop_2b_cs", "pop_2b_sb", "pop_2b_sba", "pop_2b_sba_count", "pop_3b_cs",
                          "pop_3b_sb", "pop_3b_sba", "pop_3b_sba_count", "exchange_2b_3b_sba",
                          "maxeff_arm_2b_3b_sba"],
}
BATTER_SELECTIONS = [field for group in BATTER_FIELD_GROUPS.values() for field in group]
print(len(BATTER_SELECTIONS), "batter fields in", len(BATTER_FIELD_GROUPS), "groups")

190 batter fields in 11 groups


## Savant custom leaderboard: pitchers

With `type=pitcher`, the same leaderboard returns each pitcher's own results and what batters did against them.

Here `min=0` means 0 innings. Pitchers who recorded at least one out are listed; pitchers who faced batters without recording an out are not. Position players who pitched are missing from this board but appear in expected statistics. Validation measures the gap.

`pa` is the Statcast-tracked batters-faced count and `p_total_pa` is the standard count. In the 2022-2026 check, FanGraphs' `TBF` matched `p_total_pa` for every matched pitcher, while `pa` differed by 1-2 for about 2 % of pitchers. Validation recomputes both comparisons on each run.

`p_formatted_ip` uses baseball notation: `61.1` is 61⅓ innings (184 outs) and `61.2` is 61⅔. It is stored as given and converted to outs only for calculations.

Per-pitch columns such as `{pitch}_avg_speed` and `n_{pitch}_formatted` are season summaries. They are empty, not zero, for pitches a pitcher doesn't throw. `fastball`, `breaking` and `offspeed` are group averages. `arm_angle` is in degrees.

Pitch codes: `ff` four-seam, `si` sinker, `fc` cutter, `sl` slider, `st` sweeper, `sv` slurve, `cu` curveball, `ch` changeup, `fs` splitter, `kn` knuckleball, `fo` forkball, `sc` screwball.

`fielding_run_value`, `runner_run_value`, `p_opp_batting_avg` and `p_opp_on_base_avg` are left out because they were empty on the pitcher board in every season checked. Opponent average and OBP are in `batting_avg` and `on_base_percent`.

In [5]:
# Pitch types with their own columns, then the three group averages
PITCH_CODES = ["ff", "si", "fc", "sl", "st", "sv", "cu", "ch", "fs", "kn", "fo", "sc",
               "fastball", "breaking", "offspeed"]

PITCHER_FIELD_GROUPS = {
    # arm_angle in degrees
    "profile": ["player_age", "pitch_hand", "arm_angle"],
    # p_formatted_ip is baseball innings notation
    "workload_and_record": ["p_game", "p_starting_p", "p_game_in_relief", "p_game_finished",
                            "p_complete_game", "p_shutout", "p_quality_start", "p_formatted_ip",
                            "p_out", "p_total_pa", "p_total_pitches", "p_win", "p_loss", "p_save",
                            "p_blown_save", "p_hold", "p_era", "p_earned_run", "p_unearned_run",
                            "p_run", "p_run_support", "p_inh_runner", "p_inh_runner_scored",
                            "p_beq_runner", "p_beq_runner_scored", "p_relief_no_out"],
    "outcomes_allowed": ["p_ab", "p_ab_scoring", "p_total_hits", "p_single", "p_double", "p_triple",
                         "p_home_run", "p_strikeout", "p_walk", "p_intent_walk", "p_hit_by_pitch",
                         "p_sac_bunt", "p_sac_fly", "p_total_sacrifices", "p_total_bases", "p_rbi",
                         "p_hit_scoring", "p_gnd_into_dp", "p_gnd_into_tp", "p_gnd_rule_double",
                         "p_reached_on_error", "p_catcher_interf", "p_lob", "p_walkoff"],
    "pitch_result_counts": ["p_ball", "p_called_strike", "p_foul", "p_foul_tip", "p_swinging_strike",
                            "p_total_ball", "p_total_strike", "p_total_swinging_strike",
                            "p_intent_ball", "p_automatic_ball", "p_pitchout", "p_missed_bunt",
                            "p_hit_fly", "p_hit_ground", "p_hit_into_play", "p_hit_line_drive",
                            "p_out_fly", "p_out_ground", "p_out_line_drive"],
    # while this pitcher was on the mound
    "running_game": ["p_balk", "p_wild_pitch", "p_passed_ball", "p_stolen_base_2b",
                     "p_stolen_base_3b", "p_stolen_base_home", "p_total_stolen_base",
                     "p_caught_stealing_2b", "p_caught_stealing_3b", "p_caught_stealing_home",
                     "p_total_caught_stealing", "p_defensive_indiff", "p_pickoff_1b", "p_pickoff_2b",
                     "p_pickoff_3b", "p_total_pickoff", "p_pickoff_attempt_1b", "p_pickoff_attempt_2b",
                     "p_pickoff_attempt_3b", "p_total_pickoff_attempt", "p_pickoff_error_1b",
                     "p_pickoff_error_2b", "p_pickoff_error_3b", "p_total_pickoff_error"],
    # Batter-board groups, measured against the pitcher
    "results_against": ["pa", "ab", "hit", "single", "double", "triple", "home_run", "strikeout",
                        "walk", "batting_avg", "slg_percent", "on_base_percent", "on_base_plus_slg",
                        "isolated_power", "babip", "woba", "k_percent", "bb_percent"],
    "expected_against": ["xba", "xslg", "xwoba", "xobp", "xiso", "xwobacon", "xbacon", "wobacon",
                         "bacon", "xbadiff", "xslgdiff", "wobadiff", "xera"],
    "contact_quality_against": ["batted_ball", "exit_velocity_avg", "launch_angle_avg",
                                "sweet_spot_percent", "barrel", "barrel_batted_rate",
                                "hard_hit_percent", "avg_best_speed", "avg_hyper_speed",
                                "solidcontact_percent", "flareburner_percent", "poorlyunder_percent",
                                "poorlytopped_percent", "poorlyweak_percent"],
    "batted_ball_profile_against": ["groundballs", "groundballs_percent", "flyballs",
                                    "flyballs_percent", "linedrives", "linedrives_percent", "popups",
                                    "popups_percent", "pull_percent", "straightaway_percent",
                                    "opposite_percent"],
    "plate_discipline_induced": ["pitch_count", "swing_percent", "whiff_percent", "z_swing_percent",
                                 "z_swing_miss_percent", "oz_swing_percent", "oz_swing_miss_percent",
                                 "oz_contact_percent", "iz_contact_percent", "in_zone",
                                 "in_zone_percent", "in_zone_swing", "in_zone_swing_miss", "out_zone",
                                 "out_zone_percent", "out_zone_swing", "out_zone_swing_miss",
                                 "f_strike_percent", "edge", "edge_percent", "meatball_percent",
                                 "meatball_swing_percent", "swords"],
    "bat_tracking_against": ["avg_swing_speed", "avg_swing_length", "squared_up_contact",
                             "squared_up_swing", "blasts_contact", "blasts_swing", "fast_swing_rate",
                             "attack_angle", "attack_direction", "ideal_angle_rate",
                             "vertical_swing_path"],
    "pitch_group_run_value": ["pitch_count_fastball", "pitch_count_breaking", "pitch_count_offspeed",
                              "pitch_run_value_fastball", "pitch_run_value_breaking",
                              "pitch_run_value_offspeed", "swing_take_run_value"],
    # n = pitch total behind these columns. n_{pitch}_formatted = usage %, avg_speed and range_speed = mph,
    # avg_spin = rpm, avg_break* = inches, z_induced = induced vertical break.
    "repertoire": ["n"] + [f"n_{code}_formatted" for code in PITCH_CODES] + [
        f"{code}_{stat}" for code in PITCH_CODES
        for stat in ["avg_speed", "avg_spin", "avg_break_x", "avg_break_z", "avg_break_z_induced",
                     "avg_break", "range_speed"]],
}
PITCHER_SELECTIONS = [field for group in PITCHER_FIELD_GROUPS.values() for field in group]
print(len(PITCHER_SELECTIONS), "pitcher fields in", len(PITCHER_FIELD_GROUPS), "groups")

314 pitcher fields in 13 groups


In [6]:
SAVANT_CUSTOM_URL = "https://baseballsavant.mlb.com/leaderboard/custom"
SAVANT_CUSTOM_SELECTIONS = {"batter": BATTER_SELECTIONS, "pitcher": PITCHER_SELECTIONS}
SAVANT_CUSTOM_MIN = {"batter": SAVANT_CUSTOM_MIN_BATTER, "pitcher": SAVANT_CUSTOM_MIN_PITCHER}
# Columns the CSV always adds, and schema fields not requested
SAVANT_CUSTOM_ID_COLUMNS = ["last_name, first_name", "player_id", "year"]
SAVANT_CUSTOM_NOT_REQUESTED = {
    "batter": {"player_id", "player_name", "year", "yearhidden", "xera", "r_stolen_base_pct"},
    "pitcher": {"player_id", "player_name", "year", "yearhidden", "fielding_run_value",
                "runner_run_value", "p_opp_batting_avg", "p_opp_on_base_avg"},
}


def savant_custom_schema(role, season):
    """Return Savant's custom-leaderboard field names, read from the records embedded in its page.

    The CSV export can't verify names: it returns any requested name, even a misspelled one, as an
    empty column."""
    page = fetch(SAVANT_CUSTOM_URL, {"year": season, "type": role}, f"Savant custom {role} {season} (schema)").text
    decoder = json.JSONDecoder()
    fields, records = set(), 0
    for match in re.finditer(r'"player_id"\s*:', page):
        try:
            record, _ = decoder.raw_decode(page, page.rfind("{", 0, match.start()))
        except ValueError:
            continue
        if isinstance(record, dict):
            fields |= set(record)
            records += 1
        if records == 10:
            break
    if records == 0:
        raise ValueError(f"Savant custom {role} {season}: no player records in the page, so field names could not be verified")
    return fields


def download_savant_custom(role, season):
    source = f"Savant custom {role} {season}"
    selections = SAVANT_CUSTOM_SELECTIONS[role]
    schema = savant_custom_schema(role, season)
    unknown = [field for field in selections if field not in schema]
    if unknown:
        raise ValueError(f"{source}: requested fields not in Savant's schema: {unknown}")
    unrequested = sorted(schema - set(selections) - SAVANT_CUSTOM_NOT_REQUESTED[role])
    if unrequested:
        print(f"  note: {source} schema has fields this notebook does not request yet: {unrequested}")
    params = {"year": season, "type": role, "min": SAVANT_CUSTOM_MIN[role],
              "selections": ",".join(selections), "chart": "false", "csv": "true"}
    table = read_csv_response(fetch(SAVANT_CUSTOM_URL, params, source), source)
    # A stray comma in the selections becomes an "Unnamed: n" column
    unexpected = [c for c in table.columns if c not in selections and c not in SAVANT_CUSTOM_ID_COLUMNS]
    if unexpected:
        raise ValueError(f"{source}: unexpected columns returned: {unexpected}")
    return table


def check_savant_custom(role, season):
    return lambda table: check_table(table, f"Savant custom {role}", season, "player_id", "year",
                                     SAVANT_CUSTOM_SELECTIONS[role])


savant_custom_batter, savant_custom_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_custom_batter), ("pitcher", savant_custom_pitcher)]:
        path = RAW_DIR / f"savant_custom_{role}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_savant_custom(role, season),
                                          check_savant_custom(role, season))
        register("Savant custom", role, season, path, tables[season], "player_id")

  reused savant_custom_batter_2022.csv: 693 rows x 193 columns
  reused savant_custom_pitcher_2022.csv: 813 rows x 317 columns
  reused savant_custom_batter_2023.csv: 656 rows x 193 columns
  reused savant_custom_pitcher_2023.csv: 805 rows x 317 columns
  reused savant_custom_batter_2024.csv: 651 rows x 193 columns
  reused savant_custom_pitcher_2024.csv: 806 rows x 317 columns
  reused savant_custom_batter_2025.csv: 673 rows x 193 columns
  reused savant_custom_pitcher_2025.csv: 812 rows x 317 columns
  reused savant_custom_batter_2026.csv: 662 rows x 193 columns
  reused savant_custom_pitcher_2026.csv: 819 rows x 317 columns


## Savant expected statistics

`/leaderboard/expected_statistics` returns actual and expected BA, SLG and wOBA (`est_*`, with `*_minus_*` differences), plus ERA and xERA for pitchers. Here `min` counts plate appearances.

Most values repeat custom-leaderboard fields. The table is kept because it covers different players: position players who pitched, and others the custom pitcher board omits. Validation compares the two player lists.

In [7]:
SAVANT_XSTATS_URL = "https://baseballsavant.mlb.com/leaderboard/expected_statistics"
XSTATS_REQUIRED = {
    "batter": ["last_name, first_name", "pa", "bip", "ba", "est_ba", "slg", "est_slg", "woba", "est_woba"],
    "pitcher": ["last_name, first_name", "pa", "bip", "ba", "est_ba", "slg", "est_slg", "woba", "est_woba",
                "era", "xera"],
}


def download_savant_xstats(role, season):
    source = f"Savant expected stats {role} {season}"
    params = {"type": role, "year": season, "position": "", "team": "", "min": SAVANT_XSTATS_MIN_PA, "csv": "true"}
    return read_csv_response(fetch(SAVANT_XSTATS_URL, params, source), source)


savant_xstats_batter, savant_xstats_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_xstats_batter), ("pitcher", savant_xstats_pitcher)]:
        path = RAW_DIR / f"savant_expected_stats_{role}_{season}.csv"
        validate = lambda table: check_table(table, f"Savant expected stats {role}", season,
                                             "player_id", "year", XSTATS_REQUIRED[role])
        tables[season] = load_or_download(path, lambda: download_savant_xstats(role, season), validate)
        register("Savant expected stats", role, season, path, tables[season], "player_id")

  reused savant_expected_stats_batter_2022.csv: 693 rows x 14 columns
  reused savant_expected_stats_pitcher_2022.csv: 871 rows x 17 columns
  reused savant_expected_stats_batter_2023.csv: 656 rows x 14 columns
  reused savant_expected_stats_pitcher_2023.csv: 863 rows x 17 columns
  reused savant_expected_stats_batter_2024.csv: 651 rows x 14 columns
  reused savant_expected_stats_pitcher_2024.csv: 855 rows x 17 columns


  reused savant_expected_stats_batter_2025.csv: 673 rows x 14 columns
  reused savant_expected_stats_pitcher_2025.csv: 873 rows x 17 columns


  reused savant_expected_stats_batter_2026.csv: 662 rows x 14 columns
  reused savant_expected_stats_pitcher_2026.csv: 868 rows x 17 columns


## Savant percentile rankings (supplementary)

`/leaderboard/percentile-rankings` returns rankings, not statistics. Each metric is a percentile from 1 to 100 among that season's qualified players: `xwoba = 90` means better than about 90 % of qualified batters, not an xwOBA of .900. The actual values are in the custom-leaderboard tables. The files are named `savant_percentile_ranks_*` to keep the two apart.

Savant ranks only qualified players, so many rows rank just a few metrics and many players are missing. These gaps are left as they are. Savant's export can include a placeholder row with no player name; such rows are dropped and reported.

In [8]:
SAVANT_PERCENTILE_URL = "https://baseballsavant.mlb.com/leaderboard/percentile-rankings"


def download_savant_percentiles(role, season):
    source = f"Savant percentile ranks {role} {season}"
    params = {"type": role, "year": season, "position": "", "team": "", "csv": "true"}
    table = read_csv_response(fetch(SAVANT_PERCENTILE_URL, params, source), source)
    placeholder = table["player_name"].isna()
    if placeholder.any():
        print(f"  {source}: dropped {placeholder.sum()} placeholder row(s) with no player name, "
              f"player_id {table.loc[placeholder, 'player_id'].tolist()}")
    return table.loc[~placeholder].reset_index(drop=True)


def check_percentiles(role, season):
    def validate(table):
        check_table(table, f"Savant percentile ranks {role}", season, "player_id", "year", ["player_name"])
        ranks = table.drop(columns=["player_name", "player_id", "year"]).apply(pd.to_numeric, errors="coerce")
        outside = int(((ranks < 0) | (ranks > 100)).sum().sum())
        if outside:
            raise ValueError(f"Savant percentile ranks {role} {season}: {outside} values outside 0-100")
    return validate


savant_percentiles_batter, savant_percentiles_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_percentiles_batter), ("pitcher", savant_percentiles_pitcher)]:
        path = RAW_DIR / f"savant_percentile_ranks_{role}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_savant_percentiles(role, season),
                                          check_percentiles(role, season))
        register("Savant percentile ranks", role, season, path, tables[season], "player_id")

  reused savant_percentile_ranks_batter_2022.csv: 628 rows x 23 columns


  reused savant_percentile_ranks_pitcher_2022.csv: 704 rows x 22 columns
  reused savant_percentile_ranks_batter_2023.csv: 614 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2023.csv: 707 rows x 22 columns
  reused savant_percentile_ranks_batter_2024.csv: 606 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2024.csv: 696 rows x 22 columns
  reused savant_percentile_ranks_batter_2025.csv: 617 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2025.csv: 712 rows x 22 columns
  reused savant_percentile_ranks_batter_2026.csv: 612 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2026.csv: 706 rows x 22 columns


## FanGraphs leaderboards

When this notebook was written, `pb.batting_stats()` returned an empty table with only an `IDfg` column and no error. This section requests the public leaderboard page at `/leaders/major-league` instead, with its documented URL parameters (`stats`, `season`, `qual`, `pageitems`, `pagenum`, ...).

FanGraphs embeds the leaderboard as JSON (`__NEXT_DATA__`). This code depends on that undocumented format. If it changes, the cell raises an error instead of saving an empty table. FanGraphs' robots.txt does not disallow this path. Its terms of service ask users to access the site through its provided interface; this pull loads that public page a few times per season, one request at a time.

Every column is kept under its FanGraphs name. When checked, that was 475 batting and 544 pitching fields for every `type` value tested; validation shows the current counts. They include standard and advanced stats, WAR, wRC+, FIP/xFIP/SIERA, plate discipline, batted balls, FanGraphs pitch tracking (`pfx*`, `pi*`), pitch values, Stuff+/Location+/Pitching+ (`sp_*`) and `Bats`/`Throws`. `Name` and `Team` contain HTML links; `PlayerName` and `TeamNameAbb` are plain text. `IP` uses baseball notation (186.2 is 186⅔), and `xMLBAMID` is the MLBAM ID that links rows to Savant.

`team=0` gives one row per player, with traded players combined as "2 Tms" or "3 Tms"; the `playerid` uniqueness check confirms this. `qual=0` sets no minimum, so the batting board also lists pitchers with 0 PA. `month=0` is the full season. FanGraphs leaderboards default to the regular season, and validation compares PA with Savant. Each response's query key must match the requested filters, and pages are fetched until the row count equals `totalCount`.

In [9]:
FANGRAPHS_URL = "https://www.fangraphs.com/leaders/major-league"
FANGRAPHS_PAGE_SIZE = 2000      # normally one page per season
FANGRAPHS_MAX_PAGES = 10
FANGRAPHS_REQUIRED = {"bat": ["xMLBAMID", "PlayerName", "PA", "Bats"],
                      "pit": ["xMLBAMID", "PlayerName", "IP", "TBF", "Throws"]}


def fangraphs_page(stats, season, page_number):
    source = f"FanGraphs {stats} {season} page {page_number}"
    params = {"pos": "all", "stats": stats, "lg": "all", "qual": FANGRAPHS_QUAL, "type": 8,
              "season": season, "season1": season, "month": 0, "ind": 0, "team": 0,
              "pageitems": FANGRAPHS_PAGE_SIZE, "pagenum": page_number}
    html = fetch(FANGRAPHS_URL, params, source).text
    match = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.S)
    if not match:
        raise ValueError(f"{source}: no embedded __NEXT_DATA__ block; the page layout may have changed")
    queries = json.loads(match.group(1))["props"]["pageProps"]["dehydratedState"]["queries"]
    boards = [q for q in queries if q.get("queryKey", [None])[0] == "leaders/major-league/data"]
    if len(boards) != 1:
        raise ValueError(f"{source}: expected one leaderboard dataset in the page, found {len(boards)}")
    applied, data = boards[0]["queryKey"][1], boards[0]["state"]["data"]
    # The query key echoes the filters FanGraphs applied; reject silent changes
    checked = ["stats", "season", "season1", "qual", "month", "ind", "team", "pageitems", "pagenum"]
    mismatched = {k: (applied.get(k), params[k]) for k in checked if str(applied.get(k)) != str(params[k])}
    if mismatched:
        raise ValueError(f"{source}: FanGraphs applied different filters (applied, requested): {mismatched}")
    return data["data"], data["totalCount"]


def download_fangraphs(stats, season):
    rows, total = fangraphs_page(stats, season, 1)
    page_number = 1
    while len(rows) < total:
        page_number += 1
        if page_number > FANGRAPHS_MAX_PAGES:
            raise ValueError(f"FanGraphs {stats} {season}: still incomplete after {FANGRAPHS_MAX_PAGES} pages")
        more, _ = fangraphs_page(stats, season, page_number)
        if not more:
            raise ValueError(f"FanGraphs {stats} {season}: page {page_number} was empty with {len(rows)} of {total} rows")
        rows += more
    if len(rows) != total:
        raise ValueError(f"FanGraphs {stats} {season}: collected {len(rows)} rows but totalCount is {total}")
    return pd.DataFrame(rows)


def check_fangraphs(stats, season):
    return lambda table: check_table(table, f"FanGraphs {stats}", season, "playerid", "Season",
                                     FANGRAPHS_REQUIRED[stats])


fangraphs_batting, fangraphs_pitching = {}, {}
for season in SEASONS:
    for stats, name, tables in [("bat", "batting", fangraphs_batting), ("pit", "pitching", fangraphs_pitching)]:
        path = RAW_DIR / f"fangraphs_{name}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_fangraphs(stats, season),
                                          check_fangraphs(stats, season))
        register("FanGraphs", "batter" if stats == "bat" else "pitcher", season, path, tables[season], "playerid")

  reused fangraphs_batting_2022.csv: 1495 rows x 475 columns


  reused fangraphs_pitching_2022.csv: 871 rows x 544 columns
  reused fangraphs_batting_2023.csv: 1457 rows x 475 columns


  reused fangraphs_pitching_2023.csv: 863 rows x 544 columns


  reused fangraphs_batting_2024.csv: 1454 rows x 475 columns
  reused fangraphs_pitching_2024.csv: 855 rows x 544 columns
  reused fangraphs_batting_2025.csv: 1470 rows x 475 columns
  reused fangraphs_pitching_2025.csv: 873 rows x 544 columns


  reused fangraphs_batting_2026.csv: 1472 rows x 475 columns


  reused fangraphs_pitching_2026.csv: 868 rows x 544 columns


## Player handedness (MLB Stats API)

`https://statsapi.mlb.com/api/v1/people?personIds=...` returns each player's `batSide` (L, R, or S for switch hitters) and `pitchHand`. The IDs are every MLBAM ID in the tables above, requested 100 at a time to keep URLs short.

This is the player's current listed handedness, not a record by season or plate appearance. Switch hitters are "S" even though they bat from one side in each plate appearance. Join on `player_id`, which matches Savant `player_id` and FanGraphs `xMLBAMID`.

IDs that MLB doesn't return are kept with `resolved = False` and empty handedness. On reruns, only IDs missing from the saved file are looked up and appended, and a check confirms existing rows are unchanged.

In [10]:
MLB_PEOPLE_URL = "https://statsapi.mlb.com/api/v1/people"
MLB_PEOPLE_BATCH_SIZE = 100
HANDEDNESS_COLUMNS = ["player_id", "full_name", "bat_side", "bat_side_description", "pitch_hand",
                      "pitch_hand_description", "primary_position", "resolved"]

savant_tables = [savant_custom_batter, savant_custom_pitcher, savant_xstats_batter,
                 savant_xstats_pitcher, savant_percentiles_batter, savant_percentiles_pitcher]
mlbam_ids = set()
for tables in savant_tables:
    for table in tables.values():
        mlbam_ids |= set(table["player_id"].astype(int))
for tables in [fangraphs_batting, fangraphs_pitching]:
    for table in tables.values():
        fg_ids = pd.to_numeric(table["xMLBAMID"], errors="coerce").dropna()
        mlbam_ids |= set(fg_ids[fg_ids > 0].astype(int))
mlbam_ids = sorted(mlbam_ids)
print(len(mlbam_ids), "distinct MLBAM player IDs across all season tables")


def download_handedness(player_ids):
    rows = []
    for start in range(0, len(player_ids), MLB_PEOPLE_BATCH_SIZE):
        batch = player_ids[start:start + MLB_PEOPLE_BATCH_SIZE]
        source = f"MLB people batch {start // MLB_PEOPLE_BATCH_SIZE + 1}"
        response = fetch(MLB_PEOPLE_URL, {"personIds": ",".join(map(str, batch))}, source)
        try:
            people = response.json()["people"]
        except (ValueError, KeyError) as error:
            raise ValueError(f"{source}: response was not the expected JSON ({error})") from error
        returned = [person["id"] for person in people]
        if set(returned) - set(batch) or len(returned) != len(set(returned)):
            raise ValueError(f"{source}: returned IDs that were not requested, or the same ID twice")
        for person in people:
            rows.append({"player_id": person["id"], "full_name": person.get("fullName"),
                         "bat_side": person.get("batSide", {}).get("code"),
                         "bat_side_description": person.get("batSide", {}).get("description"),
                         "pitch_hand": person.get("pitchHand", {}).get("code"),
                         "pitch_hand_description": person.get("pitchHand", {}).get("description"),
                         "primary_position": person.get("primaryPosition", {}).get("abbreviation"),
                         "resolved": True})
    resolved = {row["player_id"] for row in rows}
    rows += [{"player_id": player_id, "resolved": False} for player_id in player_ids if player_id not in resolved]
    return pd.DataFrame(rows, columns=HANDEDNESS_COLUMNS).sort_values("player_id").reset_index(drop=True)


def check_handedness(table, required_ids):
    if set(HANDEDNESS_COLUMNS) <= set(table.columns):
        problems = []
    else:
        problems = [f"missing columns {[c for c in HANDEDNESS_COLUMNS if c not in table.columns]}"]
    if not problems:
        if table.empty or table["player_id"].isna().any() or table["player_id"].duplicated().any():
            problems.append("no rows, or missing / duplicate player_id")
        uncovered = set(required_ids) - set(table["player_id"].astype(int))
        if uncovered:
            problems.append(f"{len(uncovered)} requested IDs absent, e.g. {sorted(uncovered)[:5]}")
    if problems:
        raise ValueError("handedness lookup: " + "; ".join(problems))


handedness_path = RAW_DIR / "mlb_player_handedness.csv"
if handedness_path.exists() and not OVERWRITE:
    saved = pd.read_csv(handedness_path)
    try:
        check_handedness(saved, set(saved["player_id"].dropna().astype(int)) if "player_id" in saved else set())
    except ValueError as error:
        raise ValueError(f"{handedness_path} exists but failed validation, so it was neither reused nor replaced: "
                         f"{error}. Inspect it, then delete it or set OVERWRITE = True.") from error
    new_ids = sorted(set(mlbam_ids) - set(saved["player_id"].astype(int)))
    if new_ids:
        added = download_handedness(new_ids)
        player_handedness = pd.concat([saved, added], ignore_index=True).sort_values("player_id").reset_index(drop=True)
        check_handedness(player_handedness, mlbam_ids)
        # The rows that were already saved must come through unchanged
        assert_same_values(player_handedness[player_handedness["player_id"].isin(saved["player_id"])], saved.sort_values("player_id"))
        save_csv(player_handedness, handedness_path)
        print(f"  extended {handedness_path.name}: {len(saved)} existing rows kept, {len(added)} new IDs added")
    else:
        player_handedness = saved
        print(f"  reused {handedness_path.name}: {len(saved)} rows")
else:
    player_handedness = download_handedness(mlbam_ids)
    check_handedness(player_handedness, mlbam_ids)
    save_csv(player_handedness, handedness_path)
    print(f"  saved  {handedness_path.name}: {player_handedness.shape[0]} rows x {player_handedness.shape[1]} columns")
player_handedness = pd.read_csv(handedness_path)

2635 distinct MLBAM player IDs across all season tables
  reused mlb_player_handedness.csv: 2635 rows


# Validation

These cells re-read the saved files and summarize coverage. Structural checks already ran when each file was saved or reused; these tables show the results together.

In [11]:
# Row and column counts by source, role and season
inventory = pd.DataFrame([{k: v for k, v in t.items() if k != "table"} for t in RAW_TABLES])
print("Rows")
display(inventory.pivot_table(index=["source", "role"], columns="season", values="rows", aggfunc="sum"))
print("Columns")
display(inventory.pivot_table(index=["source", "role"], columns="season", values="columns", aggfunc="sum"))
print(f"{len(inventory)} season tables + {handedness_path.name} ({len(player_handedness)} rows)")
leftover = sorted(p.name for p in RAW_DIR.glob("*.partial"))
print("unfinished .partial files:", leftover or "none")

Rows


season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter   1495  1457  1454  1470  1472
                        pitcher   871   863   855   873   868
Savant custom           batter    693   656   651   673   662
                        pitcher   813   805   806   812   819
Savant expected stats   batter    693   656   651   673   662
                        pitcher   871   863   855   873   868
Savant percentile ranks batter    628   614   606   617   612
                        pitcher   704   707   696   712   706

Columns


season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter    475   475   475   475   475
                        pitcher   544   544   544   544   544
Savant custom           batter    193   193   193   193   193
                        pitcher   317   317   317   317   317
Savant expected stats   batter     14    14    14    14    14
                        pitcher    17    17    17    17    17
Savant percentile ranks batter     23    23    23    23    23
                        pitcher    22    22    22    22    22

40 season tables + mlb_player_handedness.csv (2635 rows)
unfinished .partial files: none


In [12]:
# IDs, seasons and uniqueness, re-checked on the loaded tables
id_rows = []
for t in RAW_TABLES:
    table, id_column = t["table"], t["id_column"]
    season_column = "Season" if t["source"] == "FanGraphs" else "year"
    row = {"source": t["source"], "role": t["role"], "season": t["season"], "id_column": id_column,
           "id_dtype": str(table[id_column].dtype), "null_ids": int(table[id_column].isna().sum()),
           "duplicate_ids": int(table[id_column].duplicated().sum()),
           "seasons_in_file": ",".join(map(str, sorted(table[season_column].unique())))}
    if t["source"] == "FanGraphs":
        mlbam = pd.to_numeric(table["xMLBAMID"], errors="coerce")
        row["missing_xMLBAMID"] = int((mlbam.isna() | (mlbam <= 0)).sum())
        row["duplicate_xMLBAMID"] = int(mlbam[mlbam > 0].duplicated().sum())
    id_rows.append(row)
id_summary = pd.DataFrame(id_rows)
display(id_summary.groupby(["source", "role"]).agg(
    seasons=("seasons_in_file", ",".join), id_dtypes=("id_dtype", lambda s: ",".join(sorted(set(s)))),
    null_ids=("null_ids", "sum"), duplicate_ids=("duplicate_ids", "sum")))
fangraphs_ids = id_summary[id_summary["source"] == "FanGraphs"]
display(fangraphs_ids[["role", "season", "missing_xMLBAMID", "duplicate_xMLBAMID"]].set_index(["role", "season"]).T)
season_mismatch = id_summary[id_summary["seasons_in_file"] != id_summary["season"].astype(str)]
print("files whose season column differs from the requested season:", len(season_mismatch))

seasons id_dtypes  null_ids  \
source                  role                                                    
FanGraphs               batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant custom           batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant expected stats   batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant percentile ranks batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   

                                 duplicate_ids  
source                  role                    
FanGraphs               batter               0  
                        pitcher              0  
Savant custom           batter               0  
                        pitcher              0  
Savant expected stats   batter               0  
                        pitcher              0  
Savant percentile ranks batter               0  
                        pitcher              0

role,batter,pitcher,batter,pitcher,batter,pitcher,batter,pitcher,batter,pitcher
season,2022,2022,2023,2023,2024,2024,2025,2025,2026,2026
missing_xMLBAMID,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
duplicate_xMLBAMID,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


files whose season column differs from the requested season: 0


In [13]:
# Empty columns are reported, not rejected: some metrics don't exist in some seasons or on some boards.
# Savant field names were already checked against its schema, so these aren't typos.
missing_rows = []
for t in RAW_TABLES:
    table = t["table"]
    empty_columns = [c for c in table.columns if table[c].isna().all()]
    missing_rows.append({"source": t["source"], "role": t["role"], "season": t["season"],
                         "empty_cell_share": round(float(table.isna().mean().mean()), 3),
                         "entirely_empty_columns": len(empty_columns), "empty": empty_columns})
missing_summary = pd.DataFrame(missing_rows)
display(missing_summary.pivot_table(index=["source", "role"], columns="season",
                                    values="entirely_empty_columns", aggfunc="sum"))

# Empty columns by season, for the two broad sources
for source in ["Savant custom", "FanGraphs"]:
    for role in ["batter", "pitcher"]:
        subset = missing_summary[(missing_summary["source"] == source) & (missing_summary["role"] == role)]
        by_column = {}
        for _, row in subset.iterrows():
            for column in row["empty"]:
                by_column.setdefault(column, []).append(row["season"])
        patterns = {}
        for column, seasons in by_column.items():
            patterns.setdefault(tuple(seasons), []).append(column)
        print(f"\n{source} {role}: {len(by_column)} columns empty in at least one season")
        for seasons, columns in sorted(patterns.items(), key=lambda item: (-len(item[0]), item[0])):
            shown = ", ".join(columns[:12]) + (f", ... (+{len(columns) - 12} more)" if len(columns) > 12 else "")
            print(f"  empty in {list(seasons)}: {shown}")

season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter     80    26    26    34    25
                        pitcher    79    22    22    28    23
Savant custom           batter     12     0     0     0     0
                        pitcher    28     0     0     0     8
Savant expected stats   batter      0     0     0     0     0
                        pitcher     0     0     0     0     0
Savant percentile ranks batter      3     0     0     0     0
                        pitcher     1     1     1     1     1


Savant custom batter: 12 columns empty in at least one season
  empty in [2022]: swords, avg_swing_speed, avg_swing_length, squared_up_contact, squared_up_swing, blasts_contact, blasts_swing, fast_swing_rate, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path

Savant custom pitcher: 28 columns empty in at least one season
  empty in [2022, 2026]: n_sc_formatted, sc_avg_speed, sc_avg_spin, sc_avg_break_x, sc_avg_break_z, sc_avg_break_z_induced, sc_avg_break, sc_range_speed
  empty in [2022]: swords, avg_swing_speed, avg_swing_length, squared_up_contact, squared_up_swing, blasts_contact, blasts_swing, fast_swing_rate, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path, ... (+8 more)

FanGraphs batter: 82 columns empty in at least one season
  empty in [2022, 2023, 2024, 2025, 2026]: PO%, pfxFT%, pfxvFT, pfxFT-X, pfxFT-Z, pfxwFT, pfxwFT/C, pfxaaFT, pfxspFT
  empty in [2022, 2023, 2024, 2025]: BReview, BOverturned, BCorrect%, wBReview, PReview, POvert

In [14]:
# Savant custom field groups: share of rows with a value, averaged over each group's columns
for role, groups, tables in [("batter", BATTER_FIELD_GROUPS, savant_custom_batter),
                             ("pitcher", PITCHER_FIELD_GROUPS, savant_custom_pitcher)]:
    coverage = {}
    for season, table in tables.items():
        missing = [c for group in groups.values() for c in group if c not in table.columns]
        assert not missing, f"{role} {season}: missing requested columns {missing}"
        coverage[season] = {name: round(float(table[cols].notna().mean().mean()), 2) for name, cols in groups.items()}
    print(f"Savant custom {role}: all {sum(map(len, groups.values()))} requested columns present in every season")
    display(pd.DataFrame(coverage))

Savant custom batter: all 190 requested columns present in every season


,2022,2023,2024,2025,2026
profile,1.00,1.00,1.00,1.00,1.00
results,1.00,1.00,1.00,1.00,1.00
batting_counts,1.00,1.00,1.00,1.00,1.00
expected,0.99,0.99,0.99,0.99,0.99
contact_quality,0.99,0.99,0.99,0.99,0.99
batted_ball_profile,0.99,1.00,1.00,0.99,1.00
plate_discipline,0.95,0.99,1.00,1.00,1.00
bat_tracking,0.00,0.87,1.00,1.00,1.00
pitch_group_run_value,0.99,1.00,1.00,1.00,1.00
baserunning,0.94,0.94,0.94,0.94,0.94


Savant custom pitcher: all 314 requested columns present in every season


,2022,2023,2024,2025,2026
profile,1.00,1.00,1.00,1.00,1.00
workload_and_record,1.00,1.00,1.00,1.00,1.00
outcomes_allowed,1.00,1.00,1.00,1.00,1.00
pitch_result_counts,1.00,1.00,1.00,1.00,1.00
running_game,1.00,1.00,1.00,1.00,1.00
results_against,1.00,1.00,1.00,1.00,1.00
expected_against,1.00,1.00,1.00,1.00,1.00
contact_quality_against,1.00,1.00,1.00,1.00,1.00
batted_ball_profile_against,1.00,1.00,1.00,1.00,1.00
plate_discipline_induced,0.96,0.99,1.00,1.00,1.00


In [15]:
# FanGraphs field groups: filled share per season
FANGRAPHS_GROUPS = {
    "batter": {
        "identity_and_handedness": ["playerid", "xMLBAMID", "PlayerName", "TeamNameAbb", "Age", "Bats"],
        "standard": ["G", "PA", "AB", "H", "1B", "2B", "3B", "HR", "R", "RBI", "BB", "SO", "HBP", "SB", "AVG", "OBP", "SLG"],
        "advanced_value": ["wOBA", "wRC+", "WAR", "BABIP", "ISO", "Spd", "BaseRunning", "Offense"],
        "batted_ball": ["GB%", "LD%", "FB%", "IFFB%", "HR/FB", "Pull%", "Cent%", "Oppo%", "Soft%", "Hard%"],
        "plate_discipline": ["O-Swing%", "Z-Swing%", "Swing%", "O-Contact%", "Z-Contact%", "Contact%", "Zone%", "SwStr%", "CStr%"],
        "statcast_on_fangraphs": ["EV", "LA", "Barrel%", "HardHit%", "xwOBA", "xAVG", "xSLG"],
        "pitch_values_faced": ["wFB", "wSL", "wCB", "wCH", "wFB/C", "wSL/C"],
        "win_probability": ["WPA", "RE24", "REW", "pLI", "Clutch"],
    },
    "pitcher": {
        "identity_and_handedness": ["playerid", "xMLBAMID", "PlayerName", "TeamNameAbb", "Age", "Throws"],
        "standard": ["W", "L", "G", "GS", "SV", "HLD", "IP", "TBF", "H", "R", "ER", "HR", "BB", "SO", "ERA", "WHIP"],
        "advanced_value": ["FIP", "xFIP", "SIERA", "WAR", "K%", "BB%", "K-BB%", "LOB%", "BABIP"],
        "batted_ball": ["GB%", "LD%", "FB%", "HR/FB", "Soft%", "Hard%"],
        "plate_discipline": ["O-Swing%", "Z-Swing%", "O-Contact%", "Z-Contact%", "Zone%", "SwStr%", "C+SwStr%", "F-Strike%"],
        "statcast_on_fangraphs": ["EV", "LA", "Barrel%", "HardHit%", "xERA"],
        "pitch_mix_and_velocity": ["FB%", "FBv", "SL%", "SLv", "CH%", "CHv", "CB%", "CBv"],
        "stuff_plus": ["sp_stuff", "sp_location", "sp_pitching"],
        "win_probability": ["WPA", "RE24", "pLI", "gmLI"],
    },
}
for role, tables in [("batter", fangraphs_batting), ("pitcher", fangraphs_pitching)]:
    groups = FANGRAPHS_GROUPS[role]
    first = next(iter(tables.values()))
    absent = {name: [c for c in cols if c not in first.columns] for name, cols in groups.items()}
    absent = {name: cols for name, cols in absent.items() if cols}
    print(f"FanGraphs {role}: checked key columns absent from the source:", absent or "none")
    coverage = {season: {name: round(float(table[[c for c in cols if c in table.columns]].notna().mean().mean()), 2)
                         for name, cols in groups.items()} for season, table in tables.items()}
    display(pd.DataFrame(coverage))

FanGraphs batter: checked key columns absent from the source: none


,2022,2023,2024,2025,2026
identity_and_handedness,1.00,1.00,1.00,1.00,1.00
standard,1.00,1.00,1.00,1.00,1.00
advanced_value,0.93,0.93,0.93,0.93,0.93
batted_ball,0.73,0.72,0.72,0.73,0.72
plate_discipline,0.46,0.45,0.45,0.46,0.45
statcast_on_fangraphs,0.46,0.45,0.45,0.45,0.45
pitch_values_faced,0.46,0.44,0.44,0.45,0.44
win_probability,0.46,0.45,0.45,0.46,0.45


FanGraphs pitcher: checked key columns absent from the source: none


,2022,2023,2024,2025,2026
identity_and_handedness,1.00,1.00,1.00,1.00,1.00
standard,1.00,1.00,1.00,1.00,1.00
advanced_value,1.00,1.00,1.00,1.00,1.00
batted_ball,1.00,1.00,1.00,1.00,1.00
plate_discipline,1.00,1.00,1.00,1.00,1.00
statcast_on_fangraphs,1.00,1.00,1.00,1.00,1.00
pitch_mix_and_velocity,0.75,0.73,0.73,0.73,0.75
stuff_plus,1.00,1.00,1.00,1.00,1.00
win_probability,1.00,1.00,1.00,1.00,1.00


In [16]:
# Savant custom vs expected stats: who is on only one list. Their inclusion rules differ.
coverage_rows, xstats_only_examples = [], {}
for role, custom, xstats in [("batter", savant_custom_batter, savant_xstats_batter),
                             ("pitcher", savant_custom_pitcher, savant_xstats_pitcher)]:
    for season in SEASONS:
        custom_ids, xstats_ids = set(custom[season]["player_id"]), set(xstats[season]["player_id"])
        only_x = xstats[season][xstats[season]["player_id"].isin(xstats_ids - custom_ids)]
        positions = player_handedness.set_index("player_id").reindex(only_x["player_id"])["primary_position"]
        coverage_rows.append({"role": role, "season": season, "custom": len(custom_ids), "expected_stats": len(xstats_ids),
                              "both": len(custom_ids & xstats_ids), "custom_only": len(custom_ids - xstats_ids),
                              "expected_only": len(only_x),
                              "expected_only_pa_range": f"{only_x['pa'].min()}-{only_x['pa'].max()}" if len(only_x) else "",
                              "expected_only_non_pitchers": int((positions != "P").sum()) if role == "pitcher" else None})
        xstats_only_examples[(role, season)] = only_x
display(pd.DataFrame(coverage_rows).set_index(["role", "season"]))

custom  expected_stats  both  custom_only  expected_only  \
role    season                                                             
batter  2022       693             693   693            0              0   
        2023       656             656   656            0              0   
        2024       651             651   651            0              0   
        2025       673             673   673            0              0   
        2026       662             662   662            0              0   
pitcher 2022       813             871   813            0             58   
        2023       805             863   805            0             58   
        2024       806             855   806            0             49   
        2025       812             873   812            0             61   
        2026       819             868   819            0             49   

               expected_only_pa_range  expected_only_non_pitchers  
role    season                                                     
batter  2022                                                  NaN  
        2023                                                  NaN  
        2024                                                  NaN  
        2025                                                  NaN  
        2026                                                  NaN  
pitcher 2022                     1-47                        56.0  
        2023                     1-32                        55.0  
        2024                     1-20                        46.0  
        2025                     1-54                        61.0  
        2026                     2-49                        48.0

In [17]:
# FanGraphs vs Savant: ID overlap, and whether matched rows agree. Both are regular season with
# teams combined. PA / TBF are compared with Savant's standard counts (b_total_pa / p_total_pa) and pa.
def ip_to_outs(innings):
    """Convert baseball innings notation to outs: 61.1 is 184 outs. Only .0, .1 and .2 are valid."""
    innings = pd.to_numeric(innings)
    whole = innings // 1
    extra = ((innings - whole) * 10).round()
    if not extra.dropna().isin([0, 1, 2]).all():
        raise ValueError("found an innings value with .3 or higher, so this is not baseball innings notation")
    return whole * 3 + extra


overlap_rows = []
for role, fangraphs, savant in [("batter", fangraphs_batting, savant_custom_batter),
                                ("pitcher", fangraphs_pitching, savant_custom_pitcher)]:
    for season in SEASONS:
        fg = fangraphs[season].copy()
        fg["mlbam"] = pd.to_numeric(fg["xMLBAMID"], errors="coerce")
        if role == "batter":
            fg = fg[fg["PA"] > 0]       # the FanGraphs batting board also lists pitchers with 0 PA
        sv = savant[season]
        matched = fg.merge(sv, left_on="mlbam", right_on="player_id", how="inner", validate="one_to_one")
        row = {"role": role, "season": season, "fangraphs_rows": len(fg), "savant_rows": len(sv), "matched": len(matched),
               "fangraphs_only": int((~fg["mlbam"].isin(sv["player_id"])).sum()),
               "savant_only": int((~sv["player_id"].isin(fg["mlbam"])).sum())}
        if role == "batter":
            row["PA_eq_b_total_pa"] = round(float((matched["PA"] == matched["b_total_pa"]).mean()), 3)
            row["PA_eq_pa"] = round(float((matched["PA"] == matched["pa"]).mean()), 3)
            row["max_abs_PA_diff"] = int((matched["PA"] - matched["b_total_pa"]).abs().max())
        else:
            row["TBF_eq_p_total_pa"] = round(float((matched["TBF"] == matched["p_total_pa"]).mean()), 3)
            row["TBF_eq_pa"] = round(float((matched["TBF"] == matched["pa"]).mean()), 3)
            row["max_abs_TBF_diff"] = int((matched["TBF"] - matched["p_total_pa"]).abs().max())
            row["outs_equal_share"] = round(float((ip_to_outs(matched["IP"]) == ip_to_outs(matched["p_formatted_ip"])).mean()), 3)
        fg_only = fg[~fg["mlbam"].isin(sv["player_id"])]
        size_column = "PA" if role == "batter" else "TBF"
        row["fangraphs_only_max_" + size_column] = fg_only[size_column].max() if len(fg_only) else None
        overlap_rows.append(row)
display(pd.DataFrame(overlap_rows).set_index(["role", "season"]))

fangraphs_rows  savant_rows  matched  fangraphs_only  \
role    season                                                         
batter  2022               693          693      693               0   
        2023               656          656      656               0   
        2024               651          651      651               0   
        2025               673          673      673               0   
        2026               662          662      662               0   
pitcher 2022               871          813      813              58   
        2023               863          805      805              58   
        2024               855          806      806              49   
        2025               873          812      812              61   
        2026               868          819      819              49   

                savant_only  PA_eq_b_total_pa  PA_eq_pa  max_abs_PA_diff  \
role    season                                                             
batter  2022              0             1.000     0.993              0.0   
        2023              0             1.000     0.988              0.0   
        2024              0             1.000     1.000              0.0   
        2025              0             0.999     0.990              1.0   
        2026              0             0.997     0.985              1.0   
pitcher 2022              0               NaN       NaN              NaN   
        2023              0               NaN       NaN              NaN   
        2024              0               NaN       NaN              NaN   
        2025              0               NaN       NaN              NaN   
        2026              0               NaN       NaN              NaN   

                fangraphs_only_max_PA  TBF_eq_p_total_pa  TBF_eq_pa  \
role    season                                                        
batter  2022                      NaN                NaN        NaN   
        2023                      NaN                NaN        NaN   
        2024                      NaN                NaN        NaN   
        2025                      NaN                NaN        NaN   
        2026                      NaN                NaN        NaN   
pitcher 2022                      NaN                1.0      0.977   
        2023                      NaN                1.0      0.983   
        2024                      NaN                1.0      0.980   
        2025                      NaN                1.0      0.964   
        2026                      NaN                1.0      0.977   

                max_abs_TBF_diff  outs_equal_share  fangraphs_only_max_TBF  
role    season                                                              
batter  2022                 NaN               NaN                     NaN  
        2023                 NaN               NaN                     NaN  
        2024                 NaN               NaN                     NaN  
        2025                 NaN               NaN                     NaN  
        2026                 NaN               NaN                     NaN  
pitcher 2022                 0.0               1.0                    47.0  
        2023                 0.0               1.0                    32.0  
        2024                 0.0               1.0                    20.0  
        2025                 0.0               1.0                    54.0  
        2026                 0.0               1.0                    49.0

In [18]:
# Handedness lookup coverage, and agreement with the season tables
resolved = player_handedness["resolved"].astype(bool)
print(f"{len(player_handedness)} IDs looked up, {int(resolved.sum())} resolved, {int((~resolved).sum())} unresolved")
if (~resolved).any():
    print("  unresolved IDs:", player_handedness.loc[~resolved, "player_id"].tolist()[:20])
print("bat_side:", player_handedness["bat_side"].value_counts(dropna=False).to_dict(),
      "| pitch_hand:", player_handedness["pitch_hand"].value_counts(dropna=False).to_dict())

lookup = player_handedness.set_index("player_id")
pitch_hand = pd.concat([t[["player_id", "pitch_hand"]] for t in savant_custom_pitcher.values()]).drop_duplicates()
pitch_hand = pitch_hand.join(lookup["pitch_hand"].rename("mlb_pitch_hand"), on="player_id")
print(f"Savant pitch_hand vs MLB pitchHand: {int((pitch_hand['pitch_hand'] != pitch_hand['mlb_pitch_hand']).sum())} "
      f"disagreements in {len(pitch_hand)} pitcher rows")

# FanGraphs writes switch hitters as "B" (both); MLB uses "S"
fg_bats = pd.concat([t[["xMLBAMID", "Bats"]] for t in fangraphs_batting.values()]).drop_duplicates()
fg_bats = fg_bats.assign(Bats=fg_bats["Bats"].replace({"B": "S"})).join(lookup["bat_side"], on="xMLBAMID")
disagree = fg_bats[fg_bats["Bats"] != fg_bats["bat_side"]].dropna()
print(f"FanGraphs Bats vs MLB batSide: {len(disagree)} disagreements in {len(fg_bats)} batter rows",
      disagree.head(5).to_dict("records") if len(disagree) else "")

2635 IDs looked up, 2635 resolved, 0 unresolved


bat_side: {'R': 1695, 'L': 803, 'S': 137} | pitch_hand: {'R': 2110, 'L': 523, 'S': 2}
Savant pitch_hand vs MLB pitchHand: 0 disagreements in 1520 pitcher rows
FanGraphs Bats vs MLB batSide: 1 disagreements in 2635 batter rows [{'xMLBAMID': 666801, 'Bats': 'S', 'bat_side': 'R'}]


In [19]:
# CSV round trip: files on disk match this run's tables, and IDs read back as integers
problems = []
for t in RAW_TABLES + [{"file": handedness_path.name, "table": player_handedness, "id_column": "player_id"}]:
    on_disk = pd.read_csv(RAW_DIR / t["file"])
    try:
        assert_same_values(on_disk, t["table"])
    except AssertionError as error:
        problems.append(f"{t['file']}: {str(error)[:120]}")
    if not pd.api.types.is_integer_dtype(on_disk[t["id_column"]]):
        problems.append(f"{t['file']}: {t['id_column']} read back as {on_disk[t['id_column']].dtype}")
print(f"round-trip checked {len(RAW_TABLES) + 1} files:", problems or "all identical, IDs read back as integers")

round-trip checked 41 files: all identical, IDs read back as integers


# Game-level data

One record per regular-season game, 2022-2026. Postseason games are out of scope. MLB Stats API records supply the schedule, official results, lineups, starting pitchers, rosters, transactions and runner events. Baseball Savant supplies pitch records.

The tables keep four kinds of information apart:

| Kind | Tables | Seasons |
|---|---|---|
| Pregame game information | `games`, `lineups`, `starting_pitchers`, `bullpen_membership` | 2022-2026 |
| Actual outcomes, for evaluation only | `game_results` | 2022-2026 |
| Development play records | `plate_appearances_{season}`, `runner_movements_{season}`, `between_pitch_events`, `pitcher_appearances`, `savant_pitches/` | 2022-2025 |
| Collection checks | `schedule_entries`, `game_coverage`, `game_conflicts` | 2022-2026 |

For 2026, only pregame information and results are collected. Detailed 2026 play records aren't needed for the backtest and aren't downloaded.

Some columns in the pregame tables are known only after the game: `status`, `was_resumed`, `resumed_utc` and `first_pitch_utc` in `games`, and the cross-checks `games_started_players`, `first_listed_pitcher` and `first_batter_pitcher` in `starting_pitchers`. `schedule_entries` also carries scores. Don't use these as prediction inputs.

Join keys: `game_pk` (MLB gamePk, the same as Savant `game_pk`), `team_id` (MLB team ID) and MLBAM player IDs (`player_id`, `batter_id`, `pitcher_id`, `runner_id`; the same as Savant `player_id` and FanGraphs `xMLBAMID`). Play rows add `at_bat_index` (MLB, from 0). Savant's `at_bat_number` is checked game by game to equal `at_bat_index + 1` rather than assumed. Read the CSVs with `dtype=GAME_ID_DTYPES` so IDs with gaps stay integers.

## Game request, cache and CSV helpers

Requests go through `fetch` above: same timeout, retries, pacing and stop on access errors. Each raw response is gzipped into `GAME_CACHE_DIR`, written to `.partial` first, and parsed before it is cached, so an incomplete response is never cached. Cached responses are parsed again on every run. A cache file that fails stops the run and names the file to delete.

Game CSVs are written as soon as each partition is built: a season of play records, a month of Savant pitches, or one of the smaller tables at the end. Each is written to `.partial`, read back and compared, then renamed. If a CSV already exists and `OVERWRITE_GAMES = False`, it must match the rebuilt partition exactly or the run stops; it is never silently replaced. An interrupted run keeps its finished partitions and rebuilds the rest from the caches.

In [20]:
MLB_API = "https://statsapi.mlb.com/api"
SAVANT_STATCAST_URL = "https://baseballsavant.mlb.com/statcast_search/csv"
GAME_OUTPUT_DIR = GAME_DIR if GAME_SAMPLE is None else GAME_CACHE_DIR / "sample_output"
PLAYED_STATUSES = {"Final", "Completed Early"}   # final states of games that were played
cache_counts = Counter()


def cached_body(path, url, params, source, parse):
    """Return parse(body) for a cached response, or download, parse and cache it."""
    if path.exists():
        try:
            parsed = parse(gzip.decompress(path.read_bytes()))
        except (OSError, EOFError, ValueError) as error:
            # RuntimeError, so the collection loops stop instead of recording the game as missing
            raise RuntimeError(f"cache file {path} is unreadable or failed its check ({error}). "
                               "Delete it to download it again.") from error
        cache_counts["reused"] += 1
        return parsed
    body = fetch(url, params, source).content
    parsed = parse(body)          # raises before anything is cached
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + ".partial")
    partial.write_bytes(gzip.compress(body))
    partial.replace(path)
    cache_counts["downloaded"] += 1
    return parsed


def parse_json(*required):
    def parse(body):
        data = json.loads(body)
        missing = [key for key in required if key not in data]
        if missing:
            raise ValueError(f"JSON is missing {missing}")
        return data
    return parse


def mlb_json(cache_name, endpoint, params, source, parse):
    return cached_body(GAME_CACHE_DIR / "mlb" / cache_name, MLB_API + endpoint, params, source, parse)


def is_not_found(error):
    return isinstance(error, RuntimeError) and "HTTP 404" in str(error)


ID = "Int64"   # nullable integer, so empty IDs don't turn the column into floats
GAME_ID_DTYPES = {c: ID for c in ["game_pk", "team_id", "home_team_id", "away_team_id", "venue_id", "player_id",
                                  "batter_id", "pitcher_id", "runner_id", "winner_team_id", "basis_game_pk",
                                  "first_listed_pitcher", "first_batter_pitcher", "batting_team_id",
                                  "fielding_team_id", "at_bat_index", "play_index", "at_bat_number",
                                  "pitch_number", "batter", "pitcher", "on_1b", "on_2b", "on_3b",
                                  "start_on_1b", "start_on_2b", "start_on_3b", "pre_on_1b", "pre_on_2b",
                                  "pre_on_3b", "post_on_1b", "post_on_2b", "post_on_3b", "before_on_1b",
                                  "before_on_2b", "before_on_3b", "after_on_1b", "after_on_2b", "after_on_3b"]}
written_files = {}      # path -> "saved" or "unchanged", for the validation section


def with_id_types(table):
    return table.astype({c: t for c, t in GAME_ID_DTYPES.items() if c in table.columns})


def read_game_csv(path):
    header = pd.read_csv(path, nrows=0).columns
    return pd.read_csv(path, dtype={c: t for c, t in GAME_ID_DTYPES.items() if c in header}, low_memory=False)


def save_game_csv(table, path):
    """Write a game table safely; reuse an identical existing file, stop on a different one."""
    table = with_id_types(table).reset_index(drop=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not OVERWRITE_GAMES:
        try:
            assert_same_values(read_game_csv(path), table)
        except AssertionError as error:
            raise ValueError(f"{path} exists and differs from the rebuilt table, so it was not replaced: "
                             f"{str(error)[:200]}. Inspect it, then delete it or set OVERWRITE_GAMES = True.") from error
        written_files[path] = "unchanged"
        return
    partial = path.with_name(path.name + ".partial")
    table.to_csv(partial, index=False)
    assert_same_values(read_game_csv(partial), table)
    size_mb = partial.stat().st_size / 1e6
    if size_mb > MAX_CSV_MB:
        partial.unlink()
        raise ValueError(f"{path.name} would be {size_mb:.1f} MB, above MAX_CSV_MB = {MAX_CSV_MB}; split it further")
    partial.replace(path)
    written_files[path] = "saved"


def from_home(path):
    """Show a path under the home folder as ~/..., so printed output doesn't include a username."""
    return Path("~") / path.relative_to(Path.home()) if path.is_relative_to(Path.home()) else path


print("game cache folder:", from_home(GAME_CACHE_DIR))
print("game output folder:", GAME_OUTPUT_DIR.relative_to(REPO_ROOT) if GAME_SAMPLE is None else from_home(GAME_OUTPUT_DIR))

game cache folder: ~\.cache\mlb_game_simulator
game output folder: data\games


## Schedule and official results

`/api/v1/schedule?sportId=1&season=Y&gameType=R&hydrate=linescore,team` lists every regular-season game, one request per season. A game can be listed more than once under the same `gamePk`: postponed listings, and a suspended game's original start (`resumeDate` set) and its resumption (`resumedFrom` set). `schedule_entries` keeps every listing; `games` and `game_results` have one row per `gamePk`.

- `scheduled_start_utc` is the scheduled start of the listing that was first played. For the second game of a traditional doubleheader, MLB lists a nominal time.
- `first_pitch_utc` is the actual first pitch from the live feed's `gameInfo.firstPitch`, so it is filled for 2022-2025 only. 2026 boxscores give first pitch only as local clock text, which isn't stored.
- `game_results` holds the official final score, winner and innings played (`linescore.currentInning`). Use it only to evaluate predictions.
- Season game counts come from these records. No season is assumed to have 2,430 completed games.

In [21]:
def schedule_rows(season):
    data = mlb_json(f"schedule/{season}.json.gz", "/v1/schedule",
                    {"sportId": 1, "season": season, "gameType": "R", "hydrate": "linescore,team"},
                    f"MLB schedule {season}", parse_json("dates"))
    rows = []
    for game in (g for day in data["dates"] for g in day["games"]):
        home, away = game["teams"]["home"], game["teams"]["away"]
        if game["gameType"] != "R" or int(game["season"]) != season:
            raise ValueError(f"MLB schedule {season}: game {game['gamePk']} is {game['gameType']} {game['season']}")
        rows.append({
            "game_pk": game["gamePk"], "season": season, "official_date": game["officialDate"],
            "scheduled_start_utc": game["gameDate"], "status": game["status"]["detailedState"],
            "status_code": game["status"]["codedGameState"], "double_header": game.get("doubleHeader"),
            "game_number": game.get("gameNumber"), "day_night": game.get("dayNight"),
            "scheduled_innings": game.get("scheduledInnings"),
            "resume_scheduled_utc": game.get("resumeDate"), "resumed_from_utc": game.get("resumedFrom"),
            "reschedule_utc": game.get("rescheduleDate"), "rescheduled_from_utc": game.get("rescheduledFrom"),
            "home_team_id": home["team"]["id"], "home_team": home["team"].get("abbreviation"),
            "away_team_id": away["team"]["id"], "away_team": away["team"].get("abbreviation"),
            "venue_id": game["venue"]["id"], "venue_name": game["venue"].get("name"),
            "home_score": home.get("score"), "away_score": away.get("score"),
            "home_is_winner": home.get("isWinner"), "away_is_winner": away.get("isWinner"),
            "innings_played": game.get("linescore", {}).get("currentInning"), "is_tie": game.get("isTie")})
    return rows


schedule_entries = pd.DataFrame([row for season in GAME_SEASONS for row in schedule_rows(season)])
schedule_entries = schedule_entries.sort_values(["season", "game_pk", "scheduled_start_utc"]).reset_index(drop=True)
schedule_entries["entry"] = schedule_entries.groupby("game_pk").cumcount() + 1

game_rows, result_rows = [], []
for game_pk, entries in schedule_entries.groupby("game_pk", sort=False):
    played = entries[entries["status"].isin(PLAYED_STATUSES)]
    # A resumed game has two played listings: the original start and the resumption
    start = played[played["resumed_from_utc"].isna()] if len(played) else entries.tail(1)
    start = (start if len(start) else played).iloc[0]
    final = played.iloc[-1] if len(played) else entries.iloc[-1]
    game_rows.append({
        **{c: start[c] for c in ["game_pk", "season", "official_date", "scheduled_start_utc", "double_header",
                                 "game_number", "day_night", "scheduled_innings", "home_team_id", "home_team",
                                 "away_team_id", "away_team", "venue_id", "venue_name"]},
        "status": final["status"], "was_postponed": bool((entries["status"] == "Postponed").any()),
        "was_resumed": len(played) > 1, "resumed_utc": final["scheduled_start_utc"] if len(played) > 1 else None,
        "schedule_entries": len(entries)})
    winner = (final["home_team_id"] if final["home_is_winner"] == True       # noqa: E712 (may be NaN)
              else final["away_team_id"] if final["away_is_winner"] == True else None)
    result_rows.append({"game_pk": game_pk, "season": final["season"], "status": final["status"],
                        "home_score": final["home_score"], "away_score": final["away_score"],
                        "winner_team_id": winner, "innings_played": final["innings_played"],
                        "is_tie": final["is_tie"]})
games = pd.DataFrame(game_rows)
game_results = pd.DataFrame(result_rows)
game_results.loc[~game_results["status"].isin(PLAYED_STATUSES), ["home_score", "away_score", "innings_played"]] = None

print("regular-season games by final status:")
display(games.pivot_table(index="status", columns="season", values="game_pk", aggfunc="count", fill_value=0))
print("listings:", len(schedule_entries), "| games:", len(games),
      "| doubleheader games:", int(games["double_header"].isin(["Y", "S"]).sum()),
      "| resumed games:", int(games["was_resumed"].sum()))

regular-season games by final status:


season,2022,2023,2024,2025,2026
status,,,,,
Cancelled,0,0,1,0,1
Completed Early,9,3,4,5,2
Final,2421,2427,2425,2425,2427


listings: 12347 | games: 12150 | doubleheader games: 376 | resumed games: 16


## Games to collect

Every played game gets an MLB record. With `GAME_SAMPLE` set, only the sample games and each team's previous game are collected.

A team's previous game supplies the starting point for its bullpen (see Bullpen membership). Resumed games are skipped as a starting point: their record is final, so it can list pitchers who joined the team after the original start.

In [22]:
played_games = games[games["status"].isin(PLAYED_STATUSES)]
team_games = pd.concat([played_games.assign(team_id=played_games["home_team_id"], side="home"),
                        played_games.assign(team_id=played_games["away_team_id"], side="away")])
team_games = team_games.sort_values(["team_id", "scheduled_start_utc", "game_number"]).reset_index(drop=True)
team_games["basis_game_pk"] = (team_games["game_pk"].where(~team_games["was_resumed"])
                               .groupby([team_games["team_id"], team_games["season"]]).shift())
team_games["basis_game_pk"] = team_games.groupby(["team_id", "season"])["basis_game_pk"].ffill().astype("Int64")

if GAME_SAMPLE is None:
    target_pks = set(games["game_pk"])
else:
    unknown = sorted(set(GAME_SAMPLE) - set(games["game_pk"]))
    if unknown:
        raise ValueError(f"GAME_SAMPLE has gamePks that aren't regular-season games in GAME_SEASONS: {unknown}")
    target_pks = set(GAME_SAMPLE)
basis_pks = set(team_games.loc[team_games["game_pk"].isin(target_pks), "basis_game_pk"].dropna().astype(int))
record_games = played_games[played_games["game_pk"].isin(target_pks | basis_pks)].sort_values(["season", "game_pk"])
print(f"{len(target_pks)} target games; MLB records needed for {len(record_games)} played games "
      f"({len(set(record_games['game_pk']) - target_pks)} only as bullpen starting points)")

12150 target games; MLB records needed for 12148 played games (0 only as bullpen starting points)


## Starting lineups and starting pitchers

Both come from the game's boxscore: the live feed's `liveData.boxscore` for 2022-2025, `/api/v1/game/{gamePk}/boxscore` for 2026.

- **Lineups:** a starter's `battingOrder` is the slot times 100 (`100`-`900`) and substitutes are `101`, `102`, and so on, with `gameStatus.isSubstitute = true`. Each team must have nine distinct slots, nine distinct players and no substitutes. The order in which batters came up isn't used. `starting_position` is the first entry in MLB's `allPositions`. The boxscore's `position` field can be a position the player moved to later, so it isn't used.
- **Starting pitchers:** the player with `gamesStarted = 1`. Cross-checks: the pitcher who faced the first batter (2022-2025, from the plays), Savant's first pitch (2022-2025) and the first name in the boxscore's pitcher list. Disagreements are logged, not resolved by list order. `in_lineup` marks two-way players who both start and bat.

Only these pregame facts are kept from a game's own record. Its substitutions, relievers and starter workload are never written to these tables.

In [23]:
PITCHING_POSITION_TYPES = {"Pitcher", "Two-Way Player"}


def conflict(game_pk, team_id, check, detail):
    return {"game_pk": game_pk, "team_id": team_id, "check": check, "detail": detail}


def lineups_and_starters(game_pk, season, box, plays):
    lineup, starters, staff, problems = [], [], {}, []
    # The pitcher who faced each team's first batter
    first_faced = {}
    for play in plays or []:
        first_faced.setdefault("home" if play["about"]["isTopInning"] else "away", play["matchup"]["pitcher"]["id"])
    for side in ("away", "home"):
        team = box["teams"][side]
        team_id, players = team["team"]["id"], team["players"]
        slots = []
        for player in players.values():
            order = player.get("battingOrder")
            if order is None or int(order) % 100:
                continue
            slots.append(int(order) // 100)
            lineup.append({"game_pk": game_pk, "season": season, "team_id": team_id, "side": side,
                           "batting_slot": int(order) // 100, "player_id": player["person"]["id"],
                           # allPositions starts with the starting position; "position" can be a later one
                           "starting_position": (player.get("allPositions") or [{}])[0].get("abbreviation"),
                           "is_substitute": player.get("gameStatus", {}).get("isSubstitute")})
        team_lineup = [row for row in lineup if row["team_id"] == team_id and row["game_pk"] == game_pk]
        if sorted(slots) != list(range(1, 10)) or len({r["player_id"] for r in team_lineup}) != 9:
            problems.append(conflict(game_pk, team_id, "lineup slots", f"slots {sorted(slots)}"))
        if any(r["is_substitute"] for r in team_lineup):
            problems.append(conflict(game_pk, team_id, "lineup slots", "a slot x00 player is marked as a substitute"))

        started = [p["person"]["id"] for p in players.values()
                   if p.get("stats", {}).get("pitching", {}).get("gamesStarted") == 1]
        starter = started[0] if len(started) == 1 else None
        listed_first = team["pitchers"][0] if team["pitchers"] else None
        faced_first = first_faced.get(side)
        if starter is None:
            problems.append(conflict(game_pk, team_id, "starting pitcher", f"{len(started)} players with gamesStarted = 1"))
        for name, other in [("first listed pitcher", listed_first), ("pitcher who faced the first batter", faced_first)]:
            if starter is not None and other is not None and other != starter:
                problems.append(conflict(game_pk, team_id, "starting pitcher", f"gamesStarted {starter} vs {name} {other}"))
        starters.append({"game_pk": game_pk, "season": season, "team_id": team_id, "side": side,
                         "player_id": starter, "games_started_players": len(started),
                         "first_listed_pitcher": listed_first, "first_batter_pitcher": faced_first,
                         "in_lineup": starter in {r["player_id"] for r in team_lineup}})
        # Pitchers on the game's staff (used ones and the unused bullpen), for later games' bullpens only
        staff[team_id] = {pid for pid in set(team["pitchers"]) | set(team.get("bullpen", []))
                          if players.get(f"ID{pid}", {}).get("position", {}).get("type") in PITCHING_POSITION_TYPES}
    return lineup, starters, staff, problems

## Plate appearances and runner movements (2022-2025)

Built from the live feed (`/api/v1.1/game/{gamePk}/feed/live`). Each play lists its runner movements, and each movement's `playIndex` points to the event that caused it. Savant shows only the state before each pitch, not why runners moved.

- `plate_appearances_{season}`: one row per play, including interrupted plate appearances. Three states per row: `start_*` at the start of the plate appearance, `pre_*` just before the event that ended it, and `post_*` after it. `outcome_runs` and `outcome_outs` count only movements attributed to the batter's result.
- `runner_movements_{season}`: one row per movement leg, as MLB records it. `phase` is:
  - `batter_outcome`: on the event that ended the plate appearance.
  - `between_pitch`: steals, caught stealing, pickoffs, wild pitches, passed balls, balks, defensive indifference and runner placements on earlier events.
  - `same_pitch_baserunning`: one of those events on the final pitch, such as a wild pitch on ball four.
- `between_pitch_events`: one row per event outside the batter's result that moved a runner, with the state before and after.
- `is_interrupted`: the play has no movement for the batter. This happens when the inning ends on a baserunning out, so the batter leads off the next inning, or when the game ends during the plate appearance, for example a rain-shortened game. These rows aren't completed batter outcomes.
- Extra-inning runners are placed from MLB's `runner_placed` action.
- Checks on each play: outs against MLB's out count, the score against MLB's score after the play, bases against MLB's runners after the play (`postOn*`), and every movement must start from an occupied base. Half-innings must end with three outs unless the game ends. Failures go to `game_conflicts`.

States are rebuilt event by event, never copied from the next plate appearance, so a steal before the next pitch isn't credited to the previous batter.

In [24]:
BASES = ("1B", "2B", "3B")
# Runner events that aren't the batter's result, even on the pitch that ends the plate appearance
BASERUNNING_EVENT_PREFIXES = ("stolen_base", "caught_stealing", "pickoff", "wild_pitch", "passed_ball", "balk",
                              "defensive_indiff", "other_advance", "runner_placed")
repeated_runner_legs = Counter()     # repeated MLB runner entries dropped, by season


def snapshot(bases, outs, score, batting):
    fielding = "home" if batting == "away" else "away"
    return {"outs": outs, "on_1b": bases.get("1B"), "on_2b": bases.get("2B"), "on_3b": bases.get("3B"),
            "bat_score": score[batting], "fld_score": score[fielding]}


def apply_event(movements, bases, phase_of):
    """Apply one event's runner movements together. Returns {phase: [runs, outs]} and problems."""
    legs_by_runner, problems = {}, []
    for movement in movements:
        legs_by_runner.setdefault(movement["details"]["runner"]["id"], []).append(movement)
    ends = []
    for runner_id, legs in legs_by_runner.items():
        for first, second in zip(legs, legs[1:]):
            if first["movement"]["end"] != second["movement"]["start"]:
                problems.append(f"runner {runner_id}: movement legs don't connect")
        start = legs[0]["movement"]["start"]
        if start in BASES:
            if start not in bases:
                problems.append(f"runner {runner_id} moved from empty {start}")
            bases.pop(start, None)
        ends.append((runner_id, legs[-1]))
    totals = {}
    for runner_id, last in ends:
        tally = totals.setdefault(phase_of(last), [0, 0])
        end = last["movement"]["end"]
        if last["movement"]["isOut"]:
            tally[1] += 1
        elif end == "score":
            tally[0] += 1
        elif end in BASES:
            if end in bases:
                problems.append(f"{end} was already occupied")
            bases[end] = runner_id
    return totals, problems


def plate_appearances(game_pk, season, feed):
    pa_rows, movement_rows, event_rows, problems = [], [], [], []
    plays = feed["liveData"]["plays"]["allPlays"]
    teams = {"home": feed["gameData"]["teams"]["home"]["id"], "away": feed["gameData"]["teams"]["away"]["id"]}
    bases, outs, half = {}, 0, None
    score = {"home": 0, "away": 0}
    for play in plays:
        about, matchup = play["about"], play["matchup"]
        batting = "away" if about["isTopInning"] else "home"
        if (about["inning"], about["isTopInning"]) != half:
            if half is not None and outs != 3:
                problems.append(conflict(game_pk, None, "half-inning outs",
                                         f"inning {half[0]} {'top' if half[1] else 'bottom'} ended with {outs} outs"))
            bases, outs, half = {}, 0, (about["inning"], about["isTopInning"])
        events = {event["index"]: event for event in play["playEvents"]}
        final_index = max(events) if events else None
        batter_id = matchup["batter"]["id"]
        interrupted = not any(r["details"]["runner"]["id"] == batter_id for r in play["runners"])

        def phase_of(movement):
            index = movement["details"].get("playIndex")
            index = final_index if index is None else index
            if str(movement["details"].get("eventType", "")).startswith(BASERUNNING_EVENT_PREFIXES):
                return "between_pitch" if index != final_index or interrupted else "same_pitch_baserunning"
            return "batter_outcome" if index == final_index and not interrupted else "between_pitch"

        by_event, seen = {}, set()
        for movement in play["runners"]:
            # MLB sometimes repeats a runner's movement, at times ending at "4B" instead of "score"; keep the first copy
            leg = movement["movement"]
            key = (movement["details"]["runner"]["id"], movement["details"].get("playIndex"), leg.get("start"),
                   "score" if leg.get("end") == "4B" else leg.get("end"), leg.get("isOut"))
            if key in seen:
                repeated_runner_legs[season] += 1
                continue
            seen.add(key)
            index = movement["details"].get("playIndex")
            if index is None:
                problems.append(conflict(game_pk, None, "runner movement", f"play {play['atBatIndex']}: no playIndex"))
                index = final_index
            by_event.setdefault(index, []).append(movement)
        placements = {i: e for i, e in events.items()
                      if e.get("type") == "action" and e["details"].get("eventType") == "runner_placed"}
        for index in placements:
            by_event.setdefault(index, [])

        start = snapshot(bases, outs, score, batting)
        pre, outcome, between = None, [0, 0], 0
        for index in sorted(by_event):
            before = snapshot(bases, outs, score, batting)
            if index == final_index:
                pre = before
            if index in placements:
                runner_id = placements[index].get("player", {}).get("id")
                if "2B" in bases:
                    problems.append(conflict(game_pk, None, "runner placement", f"play {play['atBatIndex']}: 2B occupied"))
                bases["2B"] = runner_id
            totals, issues = apply_event(by_event[index], bases, phase_of)
            for phase, (runs, made) in totals.items():
                outs += made
                score[batting] += runs
                if phase == "batter_outcome":
                    outcome[0] += runs
                    outcome[1] += made
            if outs >= 3:
                # On an inning-ending force out MLB omits forced runners who didn't move, so the batter can
                # appear to reach an occupied base. Bases no longer matter once the inning is over.
                issues = [i for i in issues if not i.endswith("already occupied")]
            problems += [conflict(game_pk, None, "runner movement", f"play {play['atBatIndex']}: {i}") for i in issues]
            for leg_number, movement in enumerate(by_event[index]):
                m, d = movement["movement"], movement["details"]
                movement_rows.append({
                    "game_pk": game_pk, "season": season, "at_bat_index": play["atBatIndex"], "play_index": index,
                    "leg": leg_number, "runner_id": d["runner"]["id"], "is_batter": d["runner"]["id"] == batter_id,
                    "phase": phase_of(movement), "event_type": d.get("eventType"),
                    "movement_reason": d.get("movementReason"), "origin_base": m.get("originBase"),
                    "start_base": m.get("start"), "end_base": m.get("end"), "out_base": m.get("outBase"),
                    "is_out": m.get("isOut"), "out_number": m.get("outNumber")})
            non_outcome = [mv for mv in by_event[index] if phase_of(mv) != "batter_outcome"]
            if non_outcome or index in placements:
                between += 1
                event = events.get(index, {})
                after = snapshot(bases, outs, score, batting)
                event_rows.append({
                    "game_pk": game_pk, "season": season, "at_bat_index": play["atBatIndex"], "play_index": index,
                    "event_type": event.get("details", {}).get("eventType")
                    or (non_outcome[0]["details"].get("eventType") if non_outcome else None),
                    "play_event_type": event.get("type"), "on_final_event": index == final_index,
                    "runs": after["bat_score"] - before["bat_score"], "outs_added": after["outs"] - before["outs"],
                    **{f"before_{k}": v for k, v in before.items()}, **{f"after_{k}": v for k, v in after.items()}})
        post = snapshot(bases, outs, score, batting)
        pre = pre or post

        # Cross-checks against MLB's own play summary. A game can end during a play (for example, called for
        # rain); that last play is unfinished, has no result, and is skipped here.
        number = play["atBatIndex"]
        if not about.get("isComplete", True):
            pass
        elif outs != play["count"]["outs"]:
            problems.append(conflict(game_pk, None, "play outs", f"play {number}: rebuilt {outs}, MLB {play['count']['outs']}"))
        if about.get("isComplete", True) and (score["away"], score["home"]) != (play["result"].get("awayScore"), play["result"].get("homeScore")):
            problems.append(conflict(game_pk, None, "play score", f"play {number}: rebuilt {score['away']}-{score['home']}, "
                                     f"MLB {play['result'].get('awayScore')}-{play['result'].get('homeScore')}"))
        mlb_bases = {base for base, key in zip(BASES, ["postOnFirst", "postOnSecond", "postOnThird"]) if key in matchup}
        if about.get("isComplete", True) and outs < 3 and mlb_bases != set(bases):
            problems.append(conflict(game_pk, None, "play bases", f"play {number}: rebuilt {sorted(bases)}, MLB {sorted(mlb_bases)}"))

        pa_rows.append({
            "game_pk": game_pk, "season": season, "at_bat_index": number, "inning": about["inning"],
            "half": "top" if about["isTopInning"] else "bottom", "batting_team_id": teams[batting],
            "fielding_team_id": teams["home" if batting == "away" else "away"], "batter_id": batter_id,
            "pitcher_id": matchup["pitcher"]["id"], "bat_side": matchup.get("batSide", {}).get("code"),
            "pitch_hand": matchup.get("pitchHand", {}).get("code"), "event_type": play["result"].get("eventType"),
            "is_interrupted": interrupted, "pitches": sum(1 for e in events.values() if e.get("isPitch")),
            # no_pitch events also include dead-ball markers (code N), which aren't balls or strikes
            "automatic_balls_strikes": sum(1 for e in events.values() if e.get("type") == "no_pitch"
                                           and e["details"].get("description", "").startswith("Automatic")),
            "between_pitch_events": between, "outcome_runs": outcome[0], "outcome_outs": outcome[1],
            **{f"start_{k}": v for k, v in start.items()}, **{f"pre_{k}": v for k, v in pre.items()},
            **{f"post_{k}": v for k, v in post.items()}})
    return pa_rows, movement_rows, event_rows, problems, score

## Pitcher appearances (2022-2025)

One row per pitcher per game, in the order the pitchers appeared, from the boxscore's pitcher list and statistics. `position_type` marks position players who pitched. These are development records for estimating workload. They are never inputs for the game they describe, and they aren't collected for 2026.

In [25]:
def pitcher_appearances(game_pk, season, box):
    rows = []
    for side in ("away", "home"):
        team = box["teams"][side]
        for order, pitcher_id in enumerate(team["pitchers"], 1):
            player = team["players"].get(f"ID{pitcher_id}", {})
            stats = player.get("stats", {}).get("pitching", {})
            rows.append({"game_pk": game_pk, "season": season, "team_id": team["team"]["id"], "pitcher_id": pitcher_id,
                         "appearance_order": order, "is_starter": stats.get("gamesStarted") == 1,
                         "position_type": player.get("position", {}).get("type"),
                         "batters_faced": stats.get("battersFaced"), "outs": stats.get("outs"),
                         "pitches": stats.get("numberOfPitches")})
    return rows

## Collect and extract game records

One pass over the games, season by season: load or download each record, then extract its rows. Play rows are converted to compact tables every `GAME_BATCH_SIZE` games. Each season's `plate_appearances_{season}` and `runner_movements_{season}` are saved when the season ends and then released, so only one season of play records is in memory at a time. A short summary of each play is kept for the Savant reconciliation.

A record that can't be found (HTTP 404) or fails its completeness check is recorded in `game_coverage` and the run continues. Access errors and unreadable cache files still stop the run. A final game whose last play is unfinished is kept and noted.

In [26]:
GAME_BATCH_SIZE = 250        # games whose play rows are held as Python rows before becoming a table
PA_SUMMARY_COLUMNS = ["game_pk", "at_bat_index", "batter_id", "pitcher_id", "pitches", "automatic_balls_strikes",
                      "event_type", "is_interrupted"]


def parse_feed(game_pk):
    def parse(body):
        data = parse_json("gameData", "liveData")(body)
        plays = data["liveData"]["plays"]["allPlays"]
        if data["gameData"]["game"]["pk"] != game_pk:
            raise ValueError(f"feed is for game {data['gameData']['game']['pk']}")
        # The last play can be unfinished when a game ends during it; that is still a final record
        if data["gameData"]["status"]["abstractGameState"] != "Final" or not plays:
            raise ValueError("feed is not a final game with plays")
        return data
    return parse


def parse_boxscore(body):
    data = parse_json("teams")(body)
    if not all(data["teams"][side].get("players") for side in ("home", "away")):
        raise ValueError("boxscore has no player records")
    return data


lineup_rows, starter_rows, event_rows, appearance_rows = [], [], [], []
conflict_rows, record_problem, game_staff, first_pitch, replayed_score, boxscore_runs = [], {}, {}, {}, {}, {}
ended_during_play = set()    # final games whose last play was unfinished
pa_summaries, play_tallies = [], Counter()


def save_season_plays(season, pa_parts, movement_parts):
    """Save one season's play partitions, keep a short summary, and tally the checks."""
    if not pa_parts:
        return
    pa = pd.concat(pa_parts, ignore_index=True)
    movements = pd.concat(movement_parts, ignore_index=True)
    save_game_csv(pa, GAME_OUTPUT_DIR / f"plate_appearances_{season}.csv")
    save_game_csv(movements, GAME_OUTPUT_DIR / f"runner_movements_{season}.csv")
    pa_summaries.append(pa[PA_SUMMARY_COLUMNS].astype({"event_type": "category"}))
    appearances = pd.DataFrame([row for row in appearance_rows if row["season"] == season])
    faced = (pa[~pa["is_interrupted"]].groupby(["game_pk", "pitcher_id"]).size().rename("plays_faced").reset_index())
    workload = (with_id_types(appearances).merge(faced, on=["game_pk", "pitcher_id"], how="left")
                .fillna({"plays_faced": 0}))       # a pitcher who faced no batter has no plays
    gap = workload["batters_faced"] - workload["plays_faced"]
    # A walk after a mid-count pitching change can be charged to the earlier pitcher, while MLB's play names the reliever.
    # The two gaps then cancel within the team-game.
    offsetting = gap.ne(0) & gap.groupby([workload["game_pk"], workload["team_id"]]).transform("sum").eq(0)
    play_tallies.update({"plate appearances": len(pa), "interrupted": int(pa["is_interrupted"].sum()),
                         "with between-pitch events": int((pa["between_pitch_events"] > 0).sum()),
                         "runner movements": len(movements), "pitcher appearances": len(workload),
                         "batters faced matches plays": int(gap.eq(0).sum()),
                         "batters faced offset within team-game": int(offsetting.sum())})
    print(f"  saved {season} play records: {len(pa)} plate appearances, {len(movements)} runner movements")


pa_batch, movement_batch, pa_parts, movement_parts = [], [], [], []
batch_games, previous_season = 0, None


def end_batch():
    global batch_games
    if pa_batch:
        pa_parts.append(with_id_types(pd.DataFrame(pa_batch)))
        movement_parts.append(with_id_types(pd.DataFrame(movement_batch)))
    pa_batch.clear()
    movement_batch.clear()
    batch_games = 0


for n, game in enumerate(record_games.itertuples(index=False), 1):
    game_pk, season = int(game.game_pk), int(game.season)
    if season != previous_season:
        end_batch()
        save_season_plays(previous_season, pa_parts, movement_parts)
        pa_parts, movement_parts, previous_season = [], [], season
    if n % 250 == 0 or n == len(record_games):
        print(f"  {season}: record {n}/{len(record_games)} ({cache_counts['downloaded']} downloaded, "
              f"{cache_counts['reused']} from cache, {len(record_problem)} unavailable)")
    try:
        if season in PLAY_SEASONS:
            feed = mlb_json(f"feed/{season}/{game_pk}.json.gz", f"/v1.1/game/{game_pk}/feed/live", {},
                            f"MLB feed {game_pk}", parse_feed(game_pk))
            box, plays = feed["liveData"]["boxscore"], feed["liveData"]["plays"]["allPlays"]
        else:
            feed, plays = None, None
            box = mlb_json(f"boxscore/{season}/{game_pk}.json.gz", f"/v1/game/{game_pk}/boxscore", {},
                           f"MLB boxscore {game_pk}", parse_boxscore)
    except (RuntimeError, ValueError) as error:
        if isinstance(error, RuntimeError) and not is_not_found(error):
            raise
        record_problem[game_pk] = f"MLB record unavailable: {str(error)[:200]}"
        continue
    lineup, starters, staff, problems = lineups_and_starters(game_pk, season, box, plays)
    game_staff.update({(game_pk, team_id): members for team_id, members in staff.items()})
    if game_pk not in target_pks:
        continue
    lineup_rows += lineup
    starter_rows += starters
    conflict_rows += problems
    boxscore_runs[game_pk] = tuple(box["teams"][s].get("teamStats", {}).get("batting", {}).get("runs") for s in ("away", "home"))
    if feed is not None:
        first_pitch[game_pk] = feed["gameData"].get("gameInfo", {}).get("firstPitch")
        if not plays[-1]["about"]["isComplete"]:
            ended_during_play.add(game_pk)
        pas, movements, between, problems, score = plate_appearances(game_pk, season, feed)
        pa_batch += pas
        movement_batch += movements
        event_rows += between
        conflict_rows += problems
        replayed_score[game_pk] = (score["away"], score["home"])
        appearance_rows += pitcher_appearances(game_pk, season, box)
        batch_games += 1
        if batch_games == GAME_BATCH_SIZE:
            end_batch()
end_batch()
save_season_plays(previous_season, pa_parts, movement_parts)
del pa_parts, movement_parts

pa_summary = (pd.concat(pa_summaries, ignore_index=True) if pa_summaries
              else pd.DataFrame(columns=PA_SUMMARY_COLUMNS))
lineups = pd.DataFrame(lineup_rows).astype({"player_id": ID}).sort_values(["game_pk", "side", "batting_slot"])
starting_pitchers = (pd.DataFrame(starter_rows).sort_values(["game_pk", "side"])
                     .astype({"player_id": ID, "first_listed_pitcher": ID, "first_batter_pitcher": ID}))
games["first_pitch_utc"] = games["game_pk"].map(first_pitch)
print(f"lineup rows {len(lineups)}, starting pitchers {len(starting_pitchers)}, "
      f"plate appearances {play_tallies['plate appearances']}, runner movements {play_tallies['runner movements']}, "
      f"between-pitch events {len(event_rows)}, pitcher appearances {len(appearance_rows)}, "
      f"games ended during a play {len(ended_during_play)}")

  2022: record 250/12148 (0 downloaded, 254 from cache, 0 unavailable)


  2022: record 500/12148 (0 downloaded, 504 from cache, 0 unavailable)


  2022: record 750/12148 (0 downloaded, 754 from cache, 0 unavailable)


  2022: record 1000/12148 (0 downloaded, 1004 from cache, 0 unavailable)


  2022: record 1250/12148 (0 downloaded, 1254 from cache, 0 unavailable)


  2022: record 1500/12148 (0 downloaded, 1504 from cache, 0 unavailable)


... (43 outputs left out)

  2026: record 11750/12148 (0 downloaded, 11754 from cache, 0 unavailable)


  2026: record 12000/12148 (0 downloaded, 12004 from cache, 0 unavailable)


  2026: record 12148/12148 (0 downloaded, 12152 from cache, 0 unavailable)


lineup rows 218664, starting pitchers 24296, plate appearances 733264, runner movements 984363, between-pitch events 28718, pitcher appearances 83082, games ended during a play 4


## Savant pitch records (2022-2025)

`/statcast_search/csv?all=true&type=details` with `hfGT=R|` (regular season), one request per date with played games. Each date's original response is cached, and only `SAVANT_PITCH_COLUMNS` are saved. Field meanings are documented at https://baseballsavant.mlb.com/csv-docs. Savant's future-looking and after-the-pitch fields (for example `pitcher_days_until_next_game` and `delta_run_exp`) aren't kept.

Months are processed one at a time and saved as `savant_pitches/savant_pitches_{YYYY-MM}.csv`, where the month is the game's official date. A suspended game's resumption dates are requested with its original month, so all of a game's rows are in one file even when Savant returns them on another date. Rows are matched to games by `game_pk`.

Each month is reconciled with the MLB plays game by game:
- each `at_bat_number` should equal `at_bat_index + 1` with the same batter and pitcher;
- pitch counts should match. Savant has a row for each automatic ball or strike (an automatic intentional walk or a pitch-timer violation), while MLB records these as `no_pitch` events, counted in `automatic_balls_strikes`;
- final events should match:
  - Savant's `truncated_pa` corresponds to an interrupted MLB play;
  - an unfinished last play has no event in either source;
  - a play with no pitch, such as a pickoff before the first pitch, has no Savant row.

Unexplained differences go to `game_conflicts`.

In [27]:
SAVANT_PITCH_COLUMNS = ["game_pk", "game_date", "at_bat_number", "pitch_number", "inning", "inning_topbot",
                        "batter", "pitcher", "stand", "p_throws", "balls", "strikes", "outs_when_up",
                        "on_1b", "on_2b", "on_3b", "bat_score", "fld_score", "post_bat_score",
                        "type", "description", "events"]
SAVANT_TEXT_COLUMNS = ["game_date", "inning_topbot", "stand", "p_throws", "type", "description", "events"]
SAVANT_KEY = ["game_pk", "at_bat_number", "pitch_number"]


def savant_day(day, season):
    def parse(body):
        if "html" in body[:200].decode("utf-8", "replace").lower() or body.lstrip()[:1] == b"<":
            raise ValueError("expected CSV but received HTML")
        table = pd.read_csv(StringIO(body.decode("utf-8-sig")), low_memory=False)
        missing = [c for c in SAVANT_PITCH_COLUMNS if c not in table.columns]
        if missing:
            raise ValueError(f"missing columns {missing}")
        return table[SAVANT_PITCH_COLUMNS].astype({c: "category" for c in SAVANT_TEXT_COLUMNS})
    params = {"all": "true", "type": "details", "hfGT": "R|", "hfSea": f"{season}|", "player_type": "pitcher",
              "game_date_gt": day, "game_date_lt": day}
    return cached_body(GAME_CACHE_DIR / "savant" / "statcast" / str(season) / f"{day}.csv.gz",
                       SAVANT_STATCAST_URL, params, f"Savant pitches {day}", parse)


def concat_savant(tables):
    """Concatenate day tables, keeping the text columns as categories."""
    # A date with no pitches (e.g. a planned completion that never happened) returns only the header row
    tables = [t for t in tables if len(t)] or tables[:1]
    combined = pd.concat([t.drop(columns=SAVANT_TEXT_COLUMNS) for t in tables], ignore_index=True)
    for column in SAVANT_TEXT_COLUMNS:
        combined[column] = union_categoricals([t[column] for t in tables], ignore_order=True)
    return combined[SAVANT_PITCH_COLUMNS]


def reconcile_savant(pitches, mlb_plays):
    """Compare Savant plate appearances with MLB plays, game by game; record the result and any conflicts."""
    # MLB names the batter and pitcher at the end of the plate appearance, after any mid-count substitution
    savant_pa = (pitches.assign(events=pitches["events"].astype(object)).sort_values(SAVANT_KEY)
                 .groupby(["game_pk", "at_bat_number"])
                 .agg(batter=("batter", "last"), pitcher=("pitcher", "last"), savant_pitches=("pitch_number", "size"),
                      savant_event=("events", "last")).reset_index())
    mlb_pa = mlb_plays.assign(at_bat_number=mlb_plays["at_bat_index"] + 1,
                              event_type=mlb_plays["event_type"].astype(object))
    both = mlb_pa.merge(savant_pa, on=["game_pk", "at_bat_number"], how="outer", indicator=True)
    for game_pk, rows in both.groupby("game_pk"):
        issues = []
        mlb_only = rows[rows["_merge"] == "left_only"]
        # A play with no pitch or automatic ball/strike (e.g. a pickoff before the first pitch) has no Savant row
        unexplained = mlb_only[(mlb_only["pitches"] + mlb_only["automatic_balls_strikes"]) > 0]
        if len(unexplained):
            issues.append(f"MLB plays missing from Savant: at_bat_number {unexplained['at_bat_number'].tolist()[:10]}")
        savant_only = rows.loc[rows["_merge"] == "right_only", "at_bat_number"]
        if len(savant_only):
            issues.append(f"Savant plate appearances not in MLB plays: at_bat_number {savant_only.tolist()[:10]}")
        matched = rows[rows["_merge"] == "both"]
        misaligned = matched[(matched["batter"] != matched["batter_id"]) | (matched["pitcher"] != matched["pitcher_id"])]
        if len(misaligned):
            issues.append(f"batter or pitcher differs at at_bat_number {misaligned['at_bat_number'].tolist()[:10]}")
        events_agree = ((matched["savant_event"] == matched["event_type"])
                        | ((matched["savant_event"] == "truncated_pa") & matched["is_interrupted"])
                        # A pickoff or caught stealing that ends the inning leaves Savant's event blank
                        | (matched["savant_event"].isna() & matched["is_interrupted"])
                        | (matched["savant_event"].isna() & matched["event_type"].isna()))   # unfinished last play
        if (~events_agree).any():
            sample = matched.loc[~events_agree, ["at_bat_number", "savant_event", "event_type"]].head(3).to_dict("records")
            issues.append(f"{int((~events_agree).sum())} final events differ, e.g. {sample}")
        # Savant has a row for each automatic ball or strike; MLB records those as no_pitch events
        counts_differ = matched[matched["savant_pitches"] != matched["pitches"] + matched["automatic_balls_strikes"]]
        if len(counts_differ):
            issues.append(f"pitch counts differ at at_bat_number {counts_differ['at_bat_number'].tolist()[:10]}")
        savant_ok[game_pk] = not issues
        conflict_rows.extend(conflict(game_pk, None, "Savant vs MLB plays", issue) for issue in issues)


play_targets = games[games["game_pk"].isin(target_pks) & games["status"].isin(PLAYED_STATUSES)
                     & games["season"].isin(PLAY_SEASONS)]
month_requests = {}      # (season, month of the official date) -> dates to request and games to keep
for game in play_targets.itertuples(index=False):
    request = month_requests.setdefault((game.season, game.official_date[:7]), {"dates": set(), "games": set()})
    request["dates"].add(game.official_date)
    request["games"].add(game.game_pk)
    if game.was_resumed:
        day = pd.Timestamp(game.resumed_utc)
        request["dates"] |= {str(day.date()), str((day - pd.Timedelta(days=1)).date())}

starter_by_side = starting_pitchers.set_index(["game_pk", "side"])["player_id"]
savant_ok, savant_tallies, unknown_games = {}, Counter(), set()
for n, ((season, month), request) in enumerate(sorted(month_requests.items()), 1):
    pitches = concat_savant([savant_day(day, season) for day in sorted(request["dates"])])
    unknown_games |= set(pitches["game_pk"]) - set(games["game_pk"])
    pitches = pitches[pitches["game_pk"].isin(request["games"])]
    duplicate_rows = pitches.duplicated()     # Savant repeats some rows exactly within a day
    savant_tallies["exact duplicate rows dropped"] += int(duplicate_rows.sum())
    pitches = with_id_types(pitches[~duplicate_rows].sort_values(SAVANT_KEY).reset_index(drop=True))
    key_duplicates = pitches.duplicated(SAVANT_KEY, keep=False)
    conflict_rows += [conflict(pk, None, "Savant duplicate pitch key", "same game, plate appearance and pitch number with different values")
                      for pk in sorted(set(pitches.loc[key_duplicates, "game_pk"]))]
    reconcile_savant(pitches, pa_summary[pa_summary["game_pk"].isin(request["games"])])
    # Savant's first pitch in each half of the first inning as a starting-pitcher cross-check
    first = pitches[pitches["inning"] == 1].groupby(["game_pk", "inning_topbot"], observed=True)["pitcher"].first()
    for (game_pk, half), savant_first in first.items():
        side = "home" if half == "Top" else "away"
        starter = starter_by_side.get((game_pk, side))
        if starter is not None and not pd.isna(starter) and savant_first != starter:
            conflict_rows.append(conflict(game_pk, None, "starting pitcher", f"{side} gamesStarted {starter} vs Savant first pitch {savant_first}"))
    save_game_csv(pitches, GAME_OUTPUT_DIR / "savant_pitches" / f"savant_pitches_{month}.csv")
    savant_tallies["rows"] += len(pitches)
    savant_tallies["games"] += pitches["game_pk"].nunique()
    if n % 6 == 0 or n == len(month_requests):
        print(f"  {n}/{len(month_requests)} Savant months ({cache_counts['downloaded']} downloads so far)")
conflict_rows += [conflict(pk, None, "Savant game not in schedule", "rows dropped") for pk in sorted(unknown_games)]
print(f"Savant pitches kept: {savant_tallies['rows']} rows for {savant_tallies['games']} games; "
      f"{savant_tallies['exact duplicate rows dropped']} exact duplicate rows dropped; "
      f"reconciled for {sum(savant_ok.values())} of {len(savant_ok)} games with MLB plays")

  6/29 Savant months (0 downloads so far)


  12/29 Savant months (0 downloads so far)


  18/29 Savant months (0 downloads so far)


  24/29 Savant months (0 downloads so far)


  29/29 Savant months (0 downloads so far)
Savant pitches kept: 2855321 rows for 9719 games; 138807 exact duplicate rows dropped; reconciled for 9711 of 9719 games with MLB plays


## Bullpen membership

Bullpen membership means the pitchers on a team's active roster before a game. It doesn't say who was rested or used. For every team-game, the starting point is the pitching staff (pitchers used plus the listed bullpen) from the team's previous game, a record that existed before this game. Transactions dated from that game through this game's date are then applied.

- `membership`: `member`, `not_member` or `uncertain`.
- `basis`: `previous_game_staff`, `transaction`, or `dated_roster`. `dated_roster` is the `/teams/{id}/roster?rosterType=active&date=` snapshot used for each team's first game of a season.
- `certainty`: `previous_game_record`, `transaction_before_game_date`, `same_day_transaction`, `trade_or_claim` (destination roster not stated) or `dated_roster`.
  - Transactions carry a date but no time, so a move dated on the game day may have come before or after the game. It is marked `uncertain`, not resolved.
  - In a 2026 check the dated-roster endpoint missed a same-day call-up, so `dated_roster` rows are also not treated as verified.
- `is_starting_pitcher` marks that game's announced starter.

The game's own pitcher appearances are never used to build its bullpen. They appear only in the check below, which compares each reconstruction, and a sample of dated rosters, with the staff MLB recorded for that game. That check measures accuracy and isn't an input. Resumed games are left out of the check, because their final record can list pitchers added before the resumption. Moves dated after a game but made retroactive to it, such as some injured-list placements, can't be known in advance and show up as differences.

In [28]:
ROSTER_ADD_CODES, ROSTER_REMOVE_CODES = {"CU", "SE"}, {"OPT", "DES", "OUT", "REL", "DFA"}
TRADE_OR_CLAIM_CODES = {"TR", "CLW", "RTN"}


def team_transactions(team_id, season):
    days = schedule_entries.loc[schedule_entries["season"] == season, "official_date"]
    return mlb_json(f"transactions/{season}/{team_id}.json.gz", "/v1/transactions",
                    {"teamId": team_id, "startDate": days.min(), "endDate": days.max()},
                    f"MLB transactions {team_id} {season}", parse_json("transactions"))["transactions"]


def dated_roster_pitchers(team_id, day):
    data = mlb_json(f"roster/{day[:4]}/{team_id}_{day}.json.gz", f"/v1/teams/{team_id}/roster",
                    {"rosterType": "active", "date": day}, f"MLB roster {team_id} {day}", parse_json("roster"))
    return {p["person"]["id"] for p in data["roster"] if p.get("position", {}).get("type") in PITCHING_POSITION_TYPES}


def roster_change(transaction, team_id):
    """Classify a transaction as ("added" | "removed", certainty) for team_id, or None."""
    code, text = transaction.get("typeCode"), transaction.get("description", "").lower()
    to_team = (transaction.get("toTeam") or {}).get("id")
    from_team = (transaction.get("fromTeam") or {}).get("id")
    if code in ROSTER_ADD_CODES and to_team == team_id:
        return "added", "transaction"
    if code in ROSTER_REMOVE_CODES:
        return "removed", "transaction"
    if code == "SC":     # status changes: injured, paternity, bereavement and other lists
        if ("placed" in text or "transferred" in text) and "list" in text:
            return "removed", "transaction"
        if "activated" in text or "reinstated" in text:
            return "added", "transaction"
    if code in TRADE_OR_CLAIM_CODES:
        if to_team == team_id:
            return "added", "trade_or_claim"
        if from_team == team_id:
            return "removed", "transaction"
    return None


known_pitchers = set().union(*game_staff.values()) if game_staff else set()
target_team_games = team_games[team_games["game_pk"].isin(target_pks)]
transactions = {}
for team_id, season in sorted(set(zip(target_team_games["team_id"], target_team_games["season"]))):
    for t in team_transactions(team_id, season):
        person = t.get("person", {}).get("id")
        is_pitcher = person in known_pitchers or re.search(r"\b(RHP|LHP)\b", t.get("description", ""))
        change = roster_change(t, team_id)
        if person and is_pitcher and change:
            transactions.setdefault((team_id, season), []).append({**t, "change": change})
print(f"roster transactions kept for {len(transactions)} team-seasons:",
      sum(len(v) for v in transactions.values()))

starter_of = starting_pitchers.set_index(["game_pk", "team_id"])["player_id"].to_dict()
dates = games.set_index("game_pk")["official_date"].to_dict()
bullpen_rows = []
for row in target_team_games.itertuples(index=False):
    game_pk, team_id, season, day = int(row.game_pk), int(row.team_id), int(row.season), row.official_date
    basis_pk = None if pd.isna(row.basis_game_pk) else int(row.basis_game_pk)
    if basis_pk is not None and (basis_pk, team_id) in game_staff:
        basis_day, basis = dates[basis_pk], "previous_game_staff"
        members = {pid: ("member", basis, "previous_game_record", None) for pid in game_staff[(basis_pk, team_id)]}
    else:
        basis_day, basis = day, "dated_roster"
        members = {pid: ("member", basis, "dated_roster", None) for pid in dated_roster_pitchers(team_id, day)}
    for t in sorted(transactions.get((team_id, season), []), key=lambda t: (t["date"], t["id"])):
        if not (basis_day <= t["date"] <= day):
            continue
        pid, (change, certainty) = t["person"]["id"], t["change"]
        on_staff = pid in members and members[pid][0] == "member"
        if t["date"] == basis_day and basis == "previous_game_staff" and (change == "added") == on_staff:
            continue                # already reflected in the previous game's staff
        same_day = t["date"] == day
        if change == "added":
            status = "uncertain" if same_day or certainty == "trade_or_claim" else "member"
        else:
            status = "uncertain" if same_day else "not_member"
        detail = "same_day_transaction" if same_day else certainty if certainty == "trade_or_claim" else "transaction_before_game_date"
        members[pid] = (status, "transaction", detail, t)
    starter = starter_of.get((game_pk, team_id))
    starter = None if starter is None or pd.isna(starter) else int(starter)
    if starter is not None and starter not in members:
        members[starter] = ("member", "starting_pitcher", "announced_starter", None)
    for pid, (status, row_basis, certainty, t) in members.items():
        bullpen_rows.append({"game_pk": game_pk, "season": season, "team_id": team_id, "player_id": pid,
                             "membership": status, "basis": row_basis, "certainty": certainty,
                             "basis_game_pk": basis_pk if row_basis == "previous_game_staff" else None,
                             "transaction_type": t["typeCode"] if t else None,
                             "transaction_date": t["date"] if t else None,
                             "is_starting_pitcher": pid == starter})
bullpen_membership = (pd.DataFrame(bullpen_rows).astype({"basis_game_pk": ID})
                      .sort_values(["game_pk", "team_id", "player_id"]).reset_index(drop=True))
print(f"bullpen membership: {len(bullpen_membership)} rows for {len(target_team_games)} team-games")
display(bullpen_membership.pivot_table(index=["membership", "certainty"], columns="season", values="player_id",
                                       aggfunc="count", fill_value=0))

roster transactions kept for 150 team-seasons: 23230


bullpen membership: 338291 rows for 24296 team-games


season                                    2022   2023   2024   2025   2026
membership certainty                                                      
member     announced_starter                98    198     27     49     24
           dated_roster                    428    359    374    382    385
           previous_game_record          64013  61550  62205  62210  62192
           transaction_before_game_date    303    257    200    232    175
not_member transaction_before_game_date    288    249    262    365    306
uncertain  same_day_transaction           4344   4006   4015   3838   3856
           trade_or_claim                  200    177    247    221    256

In [29]:
# Accuracy check only: compare each reconstruction, and a sample of dated rosters, with the staff MLB recorded for that game.
# Resumed games are skipped: their record is final, so its staff can include pitchers added before the resumption.
resumed_pks = set(games.loc[games["was_resumed"], "game_pk"])
bullpen_check = {}
for (game_pk, team_id), rows in bullpen_membership.groupby(["game_pk", "team_id"]):
    staff = game_staff.get((game_pk, team_id))
    if staff is None or game_pk in resumed_pks:
        continue
    member = set(rows.loc[rows["membership"] == "member", "player_id"])
    uncertain = set(rows.loc[rows["membership"] == "uncertain", "player_id"])
    missing, extra = staff - member - uncertain, member - staff
    bullpen_check[(game_pk, team_id)] = {"matched": len(staff & member), "missing": len(missing), "extra": len(extra),
                                         "uncertain": len(uncertain), "uncertain_on_staff": len(uncertain & staff)}
    conflict_rows += [conflict(game_pk, team_id, "bullpen vs game staff", f"{kind} {pid}")
                      for kind, ids in [("missing", missing), ("extra", extra)] for pid in sorted(ids)]
check_table_ = pd.DataFrame(bullpen_check.values())
if len(check_table_):
    total = check_table_.sum()
    print(f"bullpen reconstruction vs recorded staff, {len(check_table_)} team-games: matched {total['matched']}, "
          f"missing {total['missing']}, extra {total['extra']}, uncertain {total['uncertain']} "
          f"({total['uncertain_on_staff']} of them on the staff); exact in "
          f"{int(((check_table_['missing'] == 0) & (check_table_['extra'] == 0)).sum())} team-games")
    # Share of uncertain rows on the recorded staff, by cause. A measure of how these moves usually turned out, not an input.
    uncertain_rows = bullpen_membership[(bullpen_membership["membership"] == "uncertain")
                                        & ~bullpen_membership["game_pk"].isin(resumed_pks)]
    uncertain_rows = uncertain_rows.assign(on_recorded_staff=[
        pid in game_staff.get((g, t), ()) for g, t, pid in
        zip(uncertain_rows["game_pk"], uncertain_rows["team_id"], uncertain_rows["player_id"])])
    display(uncertain_rows.groupby(["certainty", "transaction_type"])["on_recorded_staff"]
            .agg(rows="size", share_on_staff="mean").round(3))

random_pick = random.Random(23)
roster_sample = []
for season, rows in target_team_games.groupby("season"):
    keys = sorted(set(zip(rows["game_pk"], rows["team_id"])))
    roster_sample += keys if GAME_SAMPLE is not None else random_pick.sample(keys, min(ROSTER_CHECKS_PER_SEASON, len(keys)))
roster_results = []
for game_pk, team_id in roster_sample:
    staff = game_staff.get((game_pk, team_id))
    if staff is None or game_pk in resumed_pks:
        continue
    dated = dated_roster_pitchers(int(team_id), dates[game_pk])
    roster_results.append({"missing": len(staff - dated), "extra": len(dated - staff), "exact": dated == staff,
                           **{f"reconstruction_{k}": v for k, v in bullpen_check.get((game_pk, team_id), {}).items()}})
    if dated != staff:
        conflict_rows.append(conflict(game_pk, team_id, "dated roster vs game staff",
                                      f"missing {sorted(staff - dated)}, extra {sorted(dated - staff)}"))
roster_results = pd.DataFrame(roster_results)
if len(roster_results):
    print(f"dated roster on game date vs recorded staff, {len(roster_results)} sampled team-games: exact in "
          f"{int(roster_results['exact'].sum())}, pitchers missing {int(roster_results['missing'].sum())}, "
          f"extra {int(roster_results['extra'].sum())}")

bullpen reconstruction vs recorded staff, 24264 team-games: matched 311904, missing 1065, extra 3345, uncertain 21137 (6768 of them on the staff); exact in 20240 team-games


rows  share_on_staff
certainty            transaction_type                      
same_day_transaction CLW                500           0.002
                     CU                3996           0.859
                     DES               1528           0.012
                     DFA                 46           0.000
                     OPT               4130           0.021
                     OUT                797           0.001
                     REL                248           0.000
                     RTN                 80           0.012
                     SC                5617           0.364
                     SE                1395           0.827
                     TR                1699           0.012
trade_or_claim       CLW                230           0.009
                     RTN                 12           0.000
                     TR                 859           0.003

dated roster on game date vs recorded staff, 150 sampled team-games: exact in 134, pitchers missing 13, extra 4


## Game coverage and conflicts

`game_coverage` has one row per scheduled game:
- `usable`: every applicable check passed.
- `flagged`: a record exists but a check failed; the reasons are listed, with details in `game_conflicts`.
- `missing`: a played game without a usable MLB record.
- `excluded`: a game that wasn't played, such as a cancelled game.

`notes` lists expected special cases that aren't data problems: doubleheaders, resumed games, shortened games, games that ended during a plate appearance, and uncertain bullpen rows. Final scores are checked against the boxscore's team runs for every season, and against the runs rebuilt from the plays for 2022-2025.

In [30]:
game_conflicts = pd.DataFrame(conflict_rows, columns=["game_pk", "team_id", "check", "detail"]).astype({"team_id": ID})
# Bullpen comparisons measure the reconstruction; they are noted, not treated as data problems
BULLPEN_CHECKS = {"bullpen vs game staff", "dated roster vs game staff"}
flag_checks = game_conflicts[~game_conflicts["check"].isin(BULLPEN_CHECKS)]
reasons_by_game = flag_checks.groupby("game_pk")["check"].agg(lambda c: "; ".join(sorted(set(c)))).to_dict()
uncertain_games = set(bullpen_membership.loc[bullpen_membership["membership"] == "uncertain", "game_pk"])
bullpen_differs = set(game_conflicts.loc[game_conflicts["check"] == "bullpen vs game staff", "game_pk"])
results_by_game = game_results.set_index("game_pk")
coverage_rows = []
for game in games[games["game_pk"].isin(target_pks)].itertuples(index=False):
    game_pk, played = int(game.game_pk), game.status in PLAYED_STATUSES
    official = tuple(results_by_game.loc[game_pk, ["away_score", "home_score"]])
    reasons = [reasons_by_game[game_pk]] if game_pk in reasons_by_game else []
    if played and game_pk in boxscore_runs and boxscore_runs[game_pk] != official:
        reasons.append(f"boxscore runs {boxscore_runs[game_pk]} differ from schedule score {official}")
    if game_pk in replayed_score and replayed_score[game_pk] != official:
        reasons.append(f"runs rebuilt from plays {replayed_score[game_pk]} differ from official {official}")
    if played and game.season in PLAY_SEASONS and game_pk not in record_problem and not savant_ok.get(game_pk, False):
        reasons.append("Savant pitches missing or not reconciled")
    if not played:
        category, reasons = "excluded", [f"not played ({game.status})"]
    elif game_pk in record_problem:
        category, reasons = "missing", [record_problem[game_pk]]
    else:
        category = "flagged" if reasons else "usable"
    notes = [label for label, applies in [
        ("doubleheader", game.double_header in ("Y", "S")), ("resumed", game.was_resumed),
        ("shortened", game.status == "Completed Early"), ("uncertain bullpen rows", game_pk in uncertain_games),
        ("bullpen differs from recorded staff", game_pk in bullpen_differs),
        ("game ended during a plate appearance", game_pk in ended_during_play)] if applies]
    coverage_rows.append({"game_pk": game_pk, "season": game.season, "status": game.status, "category": category,
                          "reasons": "; ".join(reasons) or None, "notes": "; ".join(notes) or None,
                          "record": None if not played or game_pk in record_problem
                          else "live feed" if game.season in PLAY_SEASONS else "boxscore"})
game_coverage = pd.DataFrame(coverage_rows)
display(game_coverage.pivot_table(index="category", columns="season", values="game_pk", aggfunc="count", fill_value=0))
print("conflicts by check:", game_conflicts["check"].value_counts().to_dict() or "none")

season,2022,2023,2024,2025,2026
category,,,,,
excluded,0,0,1,0,1
flagged,7,2,4,3,1
usable,2423,2428,2425,2427,2428


conflicts by check: {'bullpen vs game staff': 4410, 'dated roster vs game staff': 16, 'starting pitcher': 9, 'Savant vs MLB plays': 8}


## Save the remaining game tables

Season play records and Savant months were saved above. The remaining tables are small and are saved here with the same checks.

In [31]:
in_scope = lambda table: table[table["game_pk"].isin(target_pks)]
GAME_TABLES = {
    "games": in_scope(games), "schedule_entries": in_scope(schedule_entries), "game_results": in_scope(game_results),
    "lineups": lineups, "starting_pitchers": starting_pitchers, "bullpen_membership": bullpen_membership,
    "between_pitch_events": pd.DataFrame(event_rows), "pitcher_appearances": pd.DataFrame(appearance_rows),
    "game_coverage": game_coverage, "game_conflicts": game_conflicts}
for name, table in GAME_TABLES.items():
    save_game_csv(table, GAME_OUTPUT_DIR / f"{name}.csv")
stale = sorted(str(p.relative_to(GAME_OUTPUT_DIR)) for p in GAME_OUTPUT_DIR.rglob("*.csv") if p not in written_files)
print(f"game CSVs: {dict(Counter(written_files.values()))}; files in the folder that this run didn't produce:", stale or "none")

game CSVs: {'saved': 11, 'unchanged': 36}; files in the folder that this run didn't produce: none


## Game-level validation

These cells re-read the game CSVs one file at a time with `GAME_ID_DTYPES` and check keys, seasons, required fields, sizes and coverage across all seasons. A partitioned table's keys can't repeat across files because each game is in exactly one partition, and that is checked too.

In [32]:
GAME_KEYS = {"games": ["game_pk"], "schedule_entries": ["game_pk", "entry"], "game_results": ["game_pk"],
             "lineups": ["game_pk", "team_id", "batting_slot"], "starting_pitchers": ["game_pk", "team_id"],
             "bullpen_membership": ["game_pk", "team_id", "player_id"],
             "between_pitch_events": ["game_pk", "at_bat_index", "play_index"],
             "pitcher_appearances": ["game_pk", "team_id", "pitcher_id"], "game_coverage": ["game_pk"],
             "game_conflicts": None, "plate_appearances": ["game_pk", "at_bat_index"],
             "runner_movements": ["game_pk", "at_bat_index", "play_index", "leg"],
             "savant_pitches": ["game_pk", "at_bat_number", "pitch_number"]}
REQUIRED = {"lineups": ["player_id"], "game_results": ["status"], "plate_appearances": ["batter_id", "pitcher_id"],
            "runner_movements": ["runner_id"], "savant_pitches": ["batter", "pitcher"]}

summary, problems, partition_of = [], [], {}
for path in sorted(written_files):
    name = re.sub(r"_(\d{4})(-\d{2})?$", "", path.stem)
    on_disk = read_game_csv(path)
    not_integer = [c for c in on_disk.columns if c in GAME_ID_DTYPES and not pd.api.types.is_integer_dtype(on_disk[c])]
    if not_integer:
        problems.append(f"{path.name}: ID columns not integer {not_integer}")
    keys = GAME_KEYS.get(name)
    if keys and on_disk.duplicated(keys).any():
        problems.append(f"{path.name}: {int(on_disk.duplicated(keys).sum())} duplicate keys {keys}")
    for column in REQUIRED.get(name, []):
        if on_disk[column].isna().any():
            problems.append(f"{path.name}: {int(on_disk[column].isna().sum())} empty {column}")
    seasons = sorted(on_disk["season"].dropna().unique().tolist()) if "season" in on_disk else []
    if name in {"plate_appearances", "runner_movements", "between_pitch_events", "pitcher_appearances"} and set(seasons) - set(PLAY_SEASONS):
        problems.append(f"{path.name}: development table has seasons {seasons}")
    if name in {"plate_appearances", "runner_movements", "savant_pitches"}:
        for game_pk in on_disk["game_pk"].unique():
            if partition_of.setdefault((name, game_pk), path.name) != path.name:
                problems.append(f"{name}: game {game_pk} is in both {partition_of[(name, game_pk)]} and {path.name}")
    summary.append({"file": str(path.relative_to(GAME_OUTPUT_DIR)), "rows": len(on_disk), "columns": on_disk.shape[1],
                    "MB": round(path.stat().st_size / 1e6, 2), "seasons": ",".join(map(str, seasons))})
    del on_disk
display(pd.DataFrame(summary))
print(f"total size of game CSVs: {sum(r['MB'] for r in summary):.1f} MB in {len(summary)} files "
      f"(largest {max((r['MB'] for r in summary), default=0)} MB)")
print("read-back problems:", problems or "none")

,file,rows,columns,MB,seasons
0,between_pitch_events.csv,28718,21,2.30,"2022,2023,2024,2025"
1,bullpen_membership.csv,338291,11,29.41,"2022,2023,2024,2025,2026"
2,game_conflicts.csv,4443,4,0.21,
3,game_coverage.csv,12150,7,0.76,"2022,2023,2024,2025,2026"
4,game_results.csv,12150,8,0.50,"2022,2023,2024,2025,2026"
5,games.csv,12150,20,1.61,"2022,2023,2024,2025,2026"
6,lineups.csv,218664,8,8.72,"2022,2023,2024,2025,2026"
7,pitcher_appearances.csv,83082,10,3.92,"2022,2023,2024,2025"
8,plate_appearances_2022.csv,182468,36,20.74,2022
9,plate_appearances_2023.csv,184506,36,21.01,2023


total size of game CSVs: 440.2 MB in 47 files (largest 29.41 MB)
read-back problems: none


In [33]:
# Coverage across all seasons: every scheduled game is accounted for
unaccounted = target_pks - set(game_coverage["game_pk"])
print("scheduled games without a coverage row:", len(unaccounted))
print("games by coverage category and season:")
display(game_coverage.pivot_table(index="category", columns="season", values="game_pk", aggfunc="count", fill_value=0))
flagged = game_coverage[game_coverage["category"].isin(["flagged", "missing"])]
if len(flagged):
    display(flagged["reasons"].str.split("; ").explode().value_counts().rename("games").to_frame())
lineup_counts = lineups.groupby(["game_pk", "team_id"]).size()
print(f"team-games with exactly 9 lineup slots: {int((lineup_counts == 9).sum())} of {len(lineup_counts)}; "
      f"starting pitchers identified: {int(starting_pitchers['player_id'].notna().sum())} of {len(starting_pitchers)}; "
      f"two-way starters in the lineup: {int(starting_pitchers['in_lineup'].sum())}")
if play_tallies["plate appearances"]:
    print(f"interrupted plate appearances: {play_tallies['interrupted']}; between-pitch events: {len(event_rows)}; "
          f"plays with a between-pitch event: {play_tallies['with between-pitch events']} of {play_tallies['plate appearances']}")
    print("pitcher appearances where boxscore batters faced equals completed MLB plays faced:",
          f"{play_tallies['batters faced matches plays']} of {play_tallies['pitcher appearances']}; "
          f"{play_tallies['batters faced offset within team-game']} more offset within their team-game")
    print("repeated MLB runner entries dropped:", dict(sorted(repeated_runner_legs.items())))
print(f"game requests this run: {cache_counts['downloaded']} downloaded, {cache_counts['reused']} reused from cache")

scheduled games without a coverage row: 0
games by coverage category and season:


season,2022,2023,2024,2025,2026
category,,,,,
excluded,0,0,1,0,1
flagged,7,2,4,3,1
usable,2423,2428,2425,2427,2428


,games
reasons,
starting pitcher,9
Savant vs MLB plays,8
Savant pitches missing or not reconciled,8


team-games with exactly 9 lineup slots: 24296 of 24296; starting pitchers identified: 24296 of 24296; two-way starters in the lineup: 74
interrupted plate appearances: 1733; between-pitch events: 28718; plays with a between-pitch event: 27931 of 733264
pitcher appearances where boxscore batters faced equals completed MLB plays faced: 83000 of 83082; 82 more offset within their team-game
repeated MLB runner entries dropped: {2022: 1, 2023: 2, 2024: 3, 2025: 1}
game requests this run: 0 downloaded, 13337 reused from cache


In [34]:
# 0 when every file is already saved and OVERWRITE = False
print(f"network requests this run: {len(requests_made)}")
if requests_made:
    print(pd.Series([s.rsplit(" ", 1)[0] if s.startswith("MLB") else " ".join(s.split()[:3]) for s in requests_made]).value_counts().to_dict())

network requests this run: 0
